# Balance des Paiements & PIB  Extraction + Transformation (FMI / IMF SDMX 3.0 API)

## Étape 1 : extraction brute · Étape 2 : transformation BOP → BDP

| | |
|---|---|
| **Source** | IMF Statistics API (SDMX 3.0), accès public. Dataflows `IMF.STA:BOP` (balance des paiements) et `IMF.RES:WEO` (PIB et agrégats macro) |
| **Périmètre** | 50 pays (voir liste), tous les indicateurs et toutes les natures d'écriture BOP/IIP |
| **Fréquence** | Annuelle |
| **Unité** | USD courant (BOP) |
| **Période** | 10 dernières années avec données réelles (dynamique par pays) |
| **Auteur** | Georges Miguel Moukoko Ekambi |


In [1]:
import os
import time
import io
import re
from pathlib import Path
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm import tqdm  # barre texte classique (plus fiable que tqdm.auto/widgets dans VS Code)
import plotly.graph_objects as go


---
## 1. Configuration

In [2]:
BASE = Path(os.path.abspath('')).parent
RAW_DIR_BOP = BASE / 'data' / 'brut_fmi'
RAW_DIR_PIB = BASE / 'data' / 'pbi'
RAW_DIR_BOP.mkdir(parents=True, exist_ok=True)
RAW_DIR_PIB.mkdir(parents=True, exist_ok=True)

API_ROOT = 'https://api.imf.org/external/sdmx/3.0'

# Dataflow Balance des paiements
AGENCY_BOP   = 'IMF.STA'
DATAFLOW_BOP = 'BOP'
VERSION_BOP  = '21.0.0'

# Dataflow PIB / macro (World Economic Outlook)
AGENCY_WEO   = 'IMF.RES'
DATAFLOW_WEO = 'WEO'
VERSION_WEO  = '9.0.0'

ANNEE_COURANTE = 2026
N_ANNEES       = 10
# Marge de sécurité : le FMI publie avec du retard (ex. Cameroun : dernière année BOP
# disponible = 2024, pas 2026). On récupère donc plus que N_ANNEES pour laisser la
# transformation (section 4+) choisir dynamiquement les 10 dernières années *avec
# des données réelles*, au lieu de figer une fenêtre calendaire qui pourrait être vide.
MARGE_ANNEES = 3
DATE_DEBUT = f"{ANNEE_COURANTE - N_ANNEES - MARGE_ANNEES}-12-31"  # borne incluse
DATE_FIN   = f"{ANNEE_COURANTE}-12-30"                             # borne incluse

FREQUENCE = 'A'    # Annuelle
UNITE     = 'USD'  # Dollar US courant (BOP)

# Plage d'etude retenue pour l'analyse (independante de la fenetre d'extraction
# ci-dessus) : filtre applique en aval sur les donnees deja telechargees, sans
# nouvel appel API. Reduire/elargir ces deux bornes ne redeclenche jamais de
# telechargement -- seul determiner_periode() (section 5) les utilise.
ANNEE_ETUDE_DEBUT = 2014
ANNEE_ETUDE_FIN   = 2023

LIMIT_SERIES          = 10000  # au-delà de ~15000, la passerelle renvoie 403 (testé)
# 600s (pas 300s) : la fenêtre élargie (MARGE_ANNEES) alourdit la réponse, certains pays
# dépassent 300s de streaming côté serveur (ReadTimeoutError observé sur le Cameroun).
TIMEOUT_S              = 600
PAUSE_ENTRE_APPELS_S   = 2     # politesse vis-à-vis de l'accès public (pas de clé API)
PAYS = {
    'SAU': 'Arabie saoudite',
    'RUS': 'Russie',
    'USA': 'États-Unis',
    'ARE': 'Émirats arabes unis',
    'CAN': 'Canada',
    'IRQ': 'Irak',
    'NOR': 'Norvège',
    'BRA': 'Brésil',
    'KAZ': 'Kazakhstan',
    'NGA': 'Nigeria',
    'KWT': 'Koweït',
    'AGO': 'Angola',
    'LBY': 'Libye',
    'OMN': 'Oman',
    'MEX': 'Mexique',
    'GUY': 'Guyana',
    'NLD': 'Pays-Bas',
    'GBR': 'Royaume-Uni',
    'DZA': 'Algérie',
    'AZE': 'Azerbaïdjan',
    'VEN': 'Venezuela',
    'COL': 'Colombie',
    'ECU': 'Équateur',
    'MYS': 'Malaisie',
    'QAT': 'Qatar',
    'IRN': 'Iran',
    'AUS': 'Australie',
    'CHN': 'Chine',
    'GAB': 'Gabon',
    'GHA': 'Ghana',
    'COG': 'République du Congo',
    'CMR': 'Cameroun',
    'GNQ': 'Guinée équatoriale',
    'SSD': 'Soudan du Sud',
    'TCD': 'Tchad',
    'SDN': 'Soudan',
    'TTO': 'Trinité-et-Tobago',
    'ARG': 'Argentine',
    'PER': 'Pérou',
    'BRN': 'Brunei',
    'IDN': 'Indonésie',
    'VNM': 'Vietnam',
    'TUN': 'Tunisie',
    'EGY': 'Égypte',
    'TUR': 'Turquie',
    'DNK': 'Danemark',
    'ITA': 'Italie',
    'ROU': 'Roumanie',
    'ALB': 'Albanie',
    'BOL': 'Bolivie',
    'CAF': 'République centrafricaine',
}

print(f"{len(PAYS)} pays enregistrés.")

51 pays enregistrés.


---
## 2. Session HTTP + fonctions d'extraction

In [3]:
def creer_session():
    """Session requests avec retries/backoff pour les erreurs transitoires (5xx, timeouts)."""
    session = requests.Session()
    retry = Retry(
        total=4,
        backoff_factor=5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=['GET'],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('https://', adapter)
    # text/csv (et non application/vnd.sdmx.data+csv) : c'est ce format qui renvoie
    # le CSV long avec les vrais codes de dimension (COUNTRY, INDICATOR, UNIT, FREQUENCY...).
    session.headers.update({'Accept': 'text/csv'})
    return session


SESSION = creer_session()

In [4]:
def url_donnees(agency, dataflow, version, code_pays):
    """
    Clé wildcardée avec des astérisques littéraux : {CODE}.*.*
    - BOP (5 dimensions COUNTRY.BOP_ACCOUNTING_ENTRY.INDICATOR.UNIT.FREQUENCY) :
      les 2 dimensions finales (UNIT, FREQUENCY) sont omises -> wildcardées aussi.
    - WEO (3 dimensions COUNTRY.INDICATOR.FREQUENCY) :
      la clé est complète, INDICATOR et FREQUENCY explicitement en '*'.
    """
    return f"{API_ROOT}/data/dataflow/{agency}/{dataflow}/{version}/{code_pays}.*.*"


def parametres_requete():
    return {
        'c[TIME_PERIOD]': f'ge:{DATE_DEBUT}+le:{DATE_FIN}',
        'attributes': 'all',
        'detail': 'full',
        'limit': str(LIMIT_SERIES),
    }


def chemin_cache_bop(code_pays):
    return RAW_DIR_BOP / f"BOP_{code_pays}.csv"


def chemin_cache_pib(code_pays):
    return RAW_DIR_PIB / f"PIB_{code_pays}.csv"


def en_cache(chemin):
    """Cache valide = fichier présent et non vide (protège contre une écriture interrompue)."""
    return chemin.exists() and chemin.stat().st_size > 0


TENTATIVES_TIMEOUT = 3  # nb d'essais en cas de timeout réseau (en plus du 1er essai)


def _telecharger(url, chemin, forcer):
    """
    Télécharge un CSV et le met en cache sur disque. Si un cache valide existe déjà
    et forcer=False, ne refait pas l'appel réseau (extraction reprenable). Écrit
    d'abord dans un fichier temporaire puis renomme, pour ne jamais laisser un
    cache tronqué en cas d'interruption.

    Retry manuel sur les timeouts réseau : urllib3.Retry (configuré dans creer_session)
    ne retente PAS un ReadTimeoutError survenant en plein milieu d'un flux chunké déjà
    commencé (constaté empiriquement : requêtes volumineuses  beaucoup d'années, tous
    indicateurs  qui dépassent parfois les 300s de streaming côté serveur). D'où ce
    retry explicite ici, en plus de celui d'urllib3 qui couvre les erreurs de connexion.
    """
    if en_cache(chemin) and not forcer:
        return chemin, 'cache'

    derniere_erreur = None
    for tentative in range(1, TENTATIVES_TIMEOUT + 2):
        try:
            reponse = SESSION.get(url, params=parametres_requete(), timeout=TIMEOUT_S)
            reponse.raise_for_status()
            break
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            derniere_erreur = e
            if tentative <= TENTATIVES_TIMEOUT:
                print(f"  Timeout réseau (essai {tentative}/{TENTATIVES_TIMEOUT + 1}) sur {chemin.name}, nouvel essai...")
            else:
                raise
    else:
        raise derniere_erreur

    chemin_tmp = chemin.with_suffix('.csv.tmp')
    chemin_tmp.write_bytes(reponse.content)
    chemin_tmp.replace(chemin)
    return chemin, 'telecharge'


def extraire_bop(code_pays, forcer=False):
    url = url_donnees(AGENCY_BOP, DATAFLOW_BOP, VERSION_BOP, code_pays)
    return _telecharger(url, chemin_cache_bop(code_pays), forcer)


def extraire_pib(code_pays, forcer=False):
    url = url_donnees(AGENCY_WEO, DATAFLOW_WEO, VERSION_WEO, code_pays)
    return _telecharger(url, chemin_cache_pib(code_pays), forcer)

---
## 3. Configuration de la transformation

Réutilise `BASE`, `RAW_DIR_BOP`, `API_ROOT`, `UNITE`, `FREQUENCE`, `ANNEE_COURANTE`, `N_ANNEES`, `MARGE_ANNEES`
déjà définis en section 1 -- pas de redéfinition.

**Période variable, pas figée** : on retient *toutes* les années avec des données réelles dans la
fenêtre [ANNEE_COURANTE - N_ANNEES - MARGE_ANNEES, ANNEE_COURANTE], sans exiger qu'elles soient
consécutives ni qu'elles atteignent N_ANNEES -- certains pays ont une publication statistique
irrégulière (ex. Congo : trou 2017-2018 suite à la crise de la dette 2016-2019 ; Gabon : seulement
2014-2015 disponibles). Calculé par pays, séparément, dans `determiner_periode()` en section 5 :
un pays « propre » (pas de trou dans la fenêtre) obtient toujours ses N_ANNEES années les plus
récentes comme avant ; un pays à publication irrégulière obtient ce qui existe réellement plutôt
que d'être exclu du pipeline.

In [5]:
DIR_BDP = BASE / 'data' / 'bdp'
DIR_REF = BASE / 'data' / 'referentiels'
DIR_BDP.mkdir(parents=True, exist_ok=True)
DIR_REF.mkdir(parents=True, exist_ok=True)

SCALE_LBL = {0: 'Units', 3: 'Thousands', 6: 'Millions', 9: 'Billions'}

CLE_COLONNES = [
    'DATASET', 'SERIES_CODE', 'OBS_MEASURE', 'COUNTRY',
    'BOP_ACCOUNTING_ENTRY', 'INDICATOR', 'UNIT', 'FREQUENCY', 'SCALE',
]

## 4. Référentiels de libellés (codes → texte clair)

`BOP_<CODE>.csv` ne contient que des codes (`A_NFA_T`, `D1_F5A`...), pas de libellé par ligne. On récupère une seule fois les référentiels complets de l'API et on les met en cache sur disque.

In [6]:
def charger_ou_recuperer_referentiel(nom_codelist, agence='IMF.STA'):
    """
    Charge le référentiel code -> libellé depuis le cache disque
    (data/referentiels/<nom>.csv), ou le récupère depuis l'API SDMX 3.0
    et le met en cache si absent.
    """
    chemin = DIR_REF / f"{nom_codelist}.csv"
    if chemin.exists() and chemin.stat().st_size > 0:
        return pd.read_csv(chemin)

    url = f"{API_ROOT}/structure/codelist/{agence}/{nom_codelist}/+"
    reponse = requests.get(url, headers={'Accept': 'application/vnd.sdmx.structure+json;version=2.0.0'}, timeout=30)
    reponse.raise_for_status()
    data = reponse.json()
    codes = data['data']['codelists'][0]['codes']

    lignes = [(c['id'], c.get('names', {}).get('en', c.get('name', ''))) for c in codes]
    df = pd.DataFrame(lignes, columns=['CODE', 'LIBELLE'])
    df.to_csv(chemin, index=False)
    return df


LIBELLES_ENTRY   = charger_ou_recuperer_referentiel('CL_BOP_ACCOUNTING_ENTRY', agence='IMF.STA')
LIBELLES_INDIC   = charger_ou_recuperer_referentiel('CL_BOP_INDICATOR', agence='IMF.STA')
LIBELLES_COUNTRY = charger_ou_recuperer_referentiel('CL_BOP_COUNTRY', agence='IMF.STA')
LIBELLES_UNIT    = charger_ou_recuperer_referentiel('CL_UNIT', agence='IMF')
LIBELLES_FREQ    = charger_ou_recuperer_referentiel('CL_FREQ', agence='IMF')

print(f"BOP_ACCOUNTING_ENTRY : {len(LIBELLES_ENTRY)} libellés")
print(f"INDICATOR            : {len(LIBELLES_INDIC)} libellés")
print(f"COUNTRY               : {len(LIBELLES_COUNTRY)} libellés")
print(f"UNIT                  : {len(LIBELLES_UNIT)} libellés")
print(f"FREQ                  : {len(LIBELLES_FREQ)} libellés")

BOP_ACCOUNTING_ENTRY : 97 libellés
INDICATOR            : 979 libellés
COUNTRY               : 345 libellés
UNIT                  : 270 libellés
FREQ                  : 34 libellés


## 5. Fonctions de transformation (période dynamique + pivot + vérification)

In [7]:
def determiner_periode(code_pays):
    """
    Détermine les années à retenir pour ce pays : toutes les années avec au moins une
    valeur USD annuelle non vide dans le brut, à l'intérieur de la plage d'étude
    [ANNEE_ETUDE_DEBUT, ANNEE_ETUDE_FIN] (section 1) -- pas la fenêtre d'extraction, qui
    reste plus large pour ne jamais avoir besoin de retelecharger si la plage d'etude
    change : seul ce filtre, purement local, en tient compte.

    Pas d'exigence de consécutivité, pas d'exigence d'atteindre N_ANNEES années :
    certains pays ont une publication statistique irrégulière (ex. Congo, crise de la
    dette et suspension du programme FMI 2016-2019 -> trou 2017-2018 ; Gabon, seulement
    2014-2015 disponibles) -- on retient ce qui existe réellement plutôt que d'exclure
    le pays. `transformer_bop_vers_bdp` garantit ensuite que `BDP_<CODE>.csv` a
    exactement une colonne par année retenue ici, pas plus, pas moins.
    """
    bop = pd.read_csv(chemin_cache_bop(code_pays), low_memory=False, dtype={'TIME_PERIOD': str})
    annuel_usd = bop[(bop['FREQUENCY'] == FREQUENCE) & (bop['UNIT'] == UNITE)]
    annees_avec_donnees = set(
        annuel_usd.loc[annuel_usd['OBS_VALUE'].notna(), 'TIME_PERIOD'].astype(str)
    )

    annees_retenues = sorted(
        (a for a in annees_avec_donnees if ANNEE_ETUDE_DEBUT <= int(a) <= ANNEE_ETUDE_FIN),
        key=int,
    )

    if not annees_retenues:
        raise ValueError(
            f"{code_pays} : aucune donnée annuelle USD trouvée dans la plage d'étude "
            f"[{ANNEE_ETUDE_DEBUT}, {ANNEE_ETUDE_FIN}]."
        )

    return annees_retenues


def transformer_bop_vers_bdp(code_pays):
    """
    Lit data/brut_fmi/BOP_<CODE>.csv, filtre USD + Annuel + les années réellement
    disponibles pour ce pays (cf. determiner_periode -- non nécessairement consécutives
    ni au nombre de N_ANNEES), met les valeurs à l'échelle, joint les libellés (entrée,
    indicateur, pays, unité, fréquence), pivote en large, écrit data/bdp/BDP_<CODE>.csv.
    Retourne (chemin_sortie, nb_series, annees_utilisees).
    """
    chemin_bop = chemin_cache_bop(code_pays)
    if not chemin_bop.exists():
        raise FileNotFoundError(f"Pas de brut pour {code_pays} : {chemin_bop}")

    annees_pays = determiner_periode(code_pays)

    bop = pd.read_csv(chemin_bop, low_memory=False, dtype={'TIME_PERIOD': str})
    f = bop[
        (bop['UNIT'] == UNITE)
        & (bop['FREQUENCY'] == FREQUENCE)
        & (bop['TIME_PERIOD'].astype(str).isin(annees_pays))
    ].copy()

    if f.empty:
        return None, 0, annees_pays

    # SERIES_CODE construit à partir des CODES bruts (avant remplacement par les libellés)
    f['SERIES_CODE'] = (
        f['COUNTRY'] + '.' + f['BOP_ACCOUNTING_ENTRY'] + '.' + f['INDICATOR']
        + '.' + f['UNIT'] + '.' + f['FREQUENCY']
    )
    f['DATASET'] = bop['STRUCTURE_ID'].iloc[0] if 'STRUCTURE_ID' in bop.columns else 'IMF.STA:BOP'
    f['OBS_MEASURE'] = 'OBS_VALUE'

    # Mise à l'échelle AVANT le pivot (division par 10**SCALE)
    f['SCALE_NUM'] = pd.to_numeric(f['SCALE'], errors='coerce').fillna(0)
    f['VAL_ECHELLE'] = pd.to_numeric(f['OBS_VALUE'], errors='coerce') / (10 ** f['SCALE_NUM'])
    f['SCALE_LBL'] = f['SCALE_NUM'].map(SCALE_LBL).fillna(f['SCALE'])

    f = f.merge(LIBELLES_ENTRY.rename(columns={'CODE': 'BOP_ACCOUNTING_ENTRY', 'LIBELLE': 'BOP_ACCOUNTING_ENTRY_LBL'}), on='BOP_ACCOUNTING_ENTRY', how='left')
    f = f.merge(LIBELLES_INDIC.rename(columns={'CODE': 'INDICATOR', 'LIBELLE': 'INDICATOR_LBL'}), on='INDICATOR', how='left')
    f = f.merge(LIBELLES_COUNTRY.rename(columns={'CODE': 'COUNTRY', 'LIBELLE': 'COUNTRY_LBL'}), on='COUNTRY', how='left')
    f = f.merge(LIBELLES_UNIT.rename(columns={'CODE': 'UNIT', 'LIBELLE': 'UNIT_LBL'}), on='UNIT', how='left')
    f = f.merge(LIBELLES_FREQ.rename(columns={'CODE': 'FREQUENCY', 'LIBELLE': 'FREQUENCY_LBL'}), on='FREQUENCY', how='left')

    cle_pivot = ['DATASET', 'SERIES_CODE', 'OBS_MEASURE', 'COUNTRY_LBL',
                 'BOP_ACCOUNTING_ENTRY_LBL', 'INDICATOR_LBL', 'UNIT_LBL', 'FREQUENCY_LBL', 'SCALE_LBL']
    pivot = f.pivot_table(index=cle_pivot, columns='TIME_PERIOD', values='VAL_ECHELLE', aggfunc='first').reset_index()
    pivot.columns = [str(c) for c in pivot.columns]
    pivot = pivot.rename(columns={
        'COUNTRY_LBL': 'COUNTRY',
        'BOP_ACCOUNTING_ENTRY_LBL': 'BOP_ACCOUNTING_ENTRY',
        'INDICATOR_LBL': 'INDICATOR',
        'UNIT_LBL': 'UNIT',
        'FREQUENCY_LBL': 'FREQUENCY',
        'SCALE_LBL': 'SCALE',
    })

    # Garantit exactement len(annees_pays) colonnes-années : si une année de annees_pays
    # n'a par miracle aucune valeur non-nulle dans f (tous les indicateurs vides cette
    # année-là), pivot_table l'omettrait silencieusement sans cette ligne.
    for a in annees_pays:
        if a not in pivot.columns:
            pivot[a] = pd.NA

    chemin_bdp = DIR_BDP / f"BDP_{code_pays}.csv"
    pivot.to_csv(chemin_bdp, index=False)
    return chemin_bdp, len(pivot), annees_pays


def verifier_filtre_annuel(code_pays, chemin_bdp, annees_pays):
    """
    Vérifie que :
      1) le brut contenait bien des périodes trimestrielles avant transformation (le test a un sens),
      2) aucune colonne de BDP_<CODE>.csv ne représente une période non-annuelle (contient 'Q' ou n'est pas un entier à 4 chiffres),
      3) toutes les colonnes années présentes sont bien dans `annees_pays` (pas de colonne hors période),
      4) il y a exactement len(annees_pays) colonnes années -- pas moins (pas de troncature silencieuse).
    Lève une AssertionError si une seule de ces conditions échoue.
    """
    bop = pd.read_csv(chemin_cache_bop(code_pays), low_memory=False, dtype={'TIME_PERIOD': str})
    n_trimestriel_brut = bop['TIME_PERIOD'].astype(str).str.contains('Q', na=False).sum()

    df_bdp = pd.read_csv(chemin_bdp)
    colonnes_annees = [c for c in df_bdp.columns if c not in CLE_COLONNES]

    colonnes_non_annuelles = [c for c in colonnes_annees if not re.fullmatch(r'\d{4}', c)]
    colonnes_hors_periode = [c for c in colonnes_annees if c not in annees_pays]
    colonnes_manquantes = [a for a in annees_pays if a not in colonnes_annees]

    print(f"--- {code_pays} (période retenue : {annees_pays[0]}-{annees_pays[-1]}) ---")
    print(f"Lignes trimestrielles dans le brut (BOP_{code_pays}.csv)      : {n_trimestriel_brut}")
    print(f"Colonnes de BDP_{code_pays}.csv                                : {colonnes_annees}")
    print(f"Colonnes non-annuelles (ex. contenant 'Q')                    : {colonnes_non_annuelles}")
    print(f"Colonnes hors période retenue                                 : {colonnes_hors_periode}")
    print(f"Colonnes manquantes (troncature ?)                            : {colonnes_manquantes}")

    assert n_trimestriel_brut > 0, "Le brut ne contient aucune période trimestrielle : le test ne prouve rien."
    assert not colonnes_non_annuelles, f"Filtre KO : colonnes non-annuelles trouvées dans BDP : {colonnes_non_annuelles}"
    assert not colonnes_hors_periode, f"Filtre KO : colonnes hors période retenue : {colonnes_hors_periode}"
    assert not colonnes_manquantes, f"TRONCATURE : {code_pays} n'a que {len(colonnes_annees)}/{len(annees_pays)} années -> manquantes : {colonnes_manquantes}"
    print(f"OK : {len(annees_pays)}/{len(annees_pays)} années présentes, filtre trimestriel/période bien appliqué.")


---
# Passage à l'échelle : les 50 pays

## 7. Extraction complète  boucle sur les 50 pays (BOP + PIB)

Chaque pays est écrit immédiatement sur disque (`data/brut_fmi/BOP_<CODE>.csv` et `data/pbi/PIB_<CODE>.csv`)
dès réception : en cas d'interruption, relancer cette cellule reprend là où l'extraction s'était arrêtée.

In [8]:
journal = []

barre = tqdm(
    PAYS.items(), total=len(PAYS), desc='Extraction BOP + PIB', unit='pays',
    mininterval=0, miniters=1, leave=True,
)
for code, nom in barre:
    barre.set_postfix_str(f"{nom} ({code})", refresh=True)
    ligne = {'code': code, 'pays': nom}

    for cle, fonction in [('bop', extraire_bop), ('pib', extraire_pib)]:
        try:
            chemin, statut = fonction(code)
            ligne[f'statut_{cle}'] = statut
            ligne[f'taille_ko_{cle}'] = chemin.stat().st_size / 1024
            ligne[f'erreur_{cle}'] = None
            if statut == 'telecharge':
                time.sleep(PAUSE_ENTRE_APPELS_S)
        except requests.RequestException as e:
            ligne[f'statut_{cle}'] = 'echec'
            ligne[f'taille_ko_{cle}'] = None
            ligne[f'erreur_{cle}'] = str(e)
            barre.write(f"ÉCHEC {cle.upper()} {nom} ({code}) : {e}")

    journal.append(ligne)

barre.close()

df_journal = pd.DataFrame(journal)
for cle in ['bop', 'pib']:
    n_cache = (df_journal[f'statut_{cle}'] == 'cache').sum()
    n_dl = (df_journal[f'statut_{cle}'] == 'telecharge').sum()
    n_echec = (df_journal[f'statut_{cle}'] == 'echec').sum()
    print(f"{cle.upper():>3} : {n_cache} depuis le cache, {n_dl} téléchargés, {n_echec} échecs.")
df_journal

Extraction BOP + PIB: 100%|██████████| 51/51 [00:00<00:00, 178.89pays/s, République centrafricaine (CAF)]


BOP : 51 depuis le cache, 0 téléchargés, 0 échecs.
PIB : 51 depuis le cache, 0 téléchargés, 0 échecs.


,code,pays,statut_bop,taille_ko_bop,erreur_bop,statut_pib,taille_ko_pib,erreur_pib
0,SAU,Arabie saoudite,cache,15474.143555,None,cache,1096.217773,None
1,RUS,Russie,cache,51275.308594,None,cache,1245.365234,None
2,USA,États-Unis,cache,41735.191406,None,cache,1302.128906,None
3,ARE,Émirats arabes unis,cache,417.916016,None,cache,960.003906,None
4,CAN,Canada,cache,22717.014648,None,cache,1306.173828,None
5,IRQ,Irak,cache,43198.259766,None,cache,778.928711,None
6,NOR,Norvège,cache,34943.081055,None,cache,1148.958008,None
7,BRA,Brésil,cache,56616.445312,None,cache,1279.036133,None
8,KAZ,Kazakhstan,cache,70738.819336,None,cache,1071.684570,None
9,NGA,Nigeria,cache,38876.619141,None,cache,1161.632812,None


---
## 8. Vérification post-extraction

In [9]:
echecs = df_journal[(df_journal['statut_bop'] == 'echec') | (df_journal['statut_pib'] == 'echec')]
print(f"Pays avec au moins un échec : {len(echecs)}")
if len(echecs):
    display(echecs)

# Pays dont l'extraction BOP a peut-être été tronquée par LIMIT_SERIES
alertes_troncature = []
for code in PAYS:
    chemin = chemin_cache_bop(code)
    if en_cache(chemin):
        n = pd.read_csv(chemin, low_memory=False)[['BOP_ACCOUNTING_ENTRY', 'INDICATOR', 'UNIT', 'FREQUENCY']].drop_duplicates().shape[0]
        if n >= LIMIT_SERIES:
            alertes_troncature.append((code, n))

if alertes_troncature:
    print(f"\nATTENTION  pays potentiellement tronqués (>= {LIMIT_SERIES} séries) : {alertes_troncature}")
else:
    print(f"\nAucun pays n'atteint LIMIT_SERIES={LIMIT_SERIES} : pas de troncature détectée.")

Pays avec au moins un échec : 0

Aucun pays n'atteint LIMIT_SERIES=10000 : pas de troncature détectée.


---
## 9. Transformation complète  tous les pays extraits

Ne traite que les pays pour lesquels `data/brut_fmi/BOP_<CODE>.csv` existe déjà (produit par la section 7 ci-dessus).

In [10]:
FORCER_TRANSFO = False  # a repasser a True ponctuellement si BDP_<CODE>.csv doit etre reconstruit (ex. determiner_periode() ou le schema source change) ; sinon les fichiers en cache sont reutilises

fichiers_bop = sorted(RAW_DIR_BOP.glob('BOP_*.csv'))
codes_disponibles = [p.stem.replace('BOP_', '') for p in fichiers_bop]
print(f"{len(codes_disponibles)} pays disponibles en brut : {codes_disponibles}")

resultats_transfo = []
barre = tqdm(codes_disponibles, desc='Transformation BOP -> BDP', unit='pays', mininterval=0, miniters=1)
for code in barre:
    barre.set_postfix_str(code, refresh=True)
    chemin_existant = DIR_BDP / f"BDP_{code}.csv"

    if not FORCER_TRANSFO and chemin_existant.exists() and chemin_existant.stat().st_size > 0:
        n = len(pd.read_csv(chemin_existant))
        resultats_transfo.append({'code': code, 'n_series': n, 'periode': None, 'statut': 'deja_fait'})
        continue

    try:
        chemin, n, annees_pays = transformer_bop_vers_bdp(code)
        resultats_transfo.append({
            'code': code, 'n_series': n,
            'periode': f"{annees_pays[0]}-{annees_pays[-1]}",
            'statut': 'ok',
        })
    except Exception as e:
        resultats_transfo.append({'code': code, 'n_series': None, 'periode': None, 'statut': f'echec: {e}'})
        barre.write(f"ÉCHEC {code} : {e}")

df_resultats_transfo = pd.DataFrame(resultats_transfo)
for statut in ['deja_fait', 'ok']:
    n = (df_resultats_transfo['statut'] == statut).sum()
    print(f"{statut} : {n}")
df_resultats_transfo

51 pays disponibles en brut : ['AGO', 'ALB', 'ARE', 'ARG', 'AUS', 'AZE', 'BOL', 'BRA', 'BRN', 'CAF', 'CAN', 'CHN', 'CMR', 'COG', 'COL', 'DNK', 'DZA', 'ECU', 'EGY', 'GAB', 'GBR', 'GHA', 'GNQ', 'GUY', 'IDN', 'IRN', 'IRQ', 'ITA', 'KAZ', 'KWT', 'LBY', 'MEX', 'MYS', 'NGA', 'NLD', 'NOR', 'OMN', 'PER', 'QAT', 'ROU', 'RUS', 'SAU', 'SDN', 'SSD', 'TCD', 'TTO', 'TUN', 'TUR', 'USA', 'VEN', 'VNM']


Transformation BOP -> BDP:  20%|█▉        | 10/51 [00:12<00:31,  1.31pays/s, CAN]

ÉCHEC CAF : CAF : aucune donnée annuelle USD trouvée dans la plage d'étude [2014, 2023].


Transformation BOP -> BDP:  45%|████▌     | 23/51 [00:22<00:17,  1.59pays/s, GUY]

ÉCHEC GNQ : GNQ : aucune donnée annuelle USD trouvée dans la plage d'étude [2014, 2023].


...

Transformation BOP -> BDP:  88%|████████▊ | 45/51 [00:50<00:04,  1.20pays/s, TTO]

ÉCHEC TCD : TCD : aucune donnée annuelle USD trouvée dans la plage d'étude [2014, 2023].


Transformation BOP -> BDP: 100%|██████████| 51/51 [00:56<00:00,  1.12s/pays, VNM]

deja_fait : 0
ok : 47


,code,n_series,periode,statut
0,AGO,1172.0,2014-2023,ok
1,ALB,863.0,2014-2023,ok
2,ARE,101.0,2023-2023,ok
3,ARG,1120.0,2014-2023,ok
4,AUS,501.0,2014-2023,ok
5,AZE,1171.0,2014-2023,ok
6,BOL,891.0,2014-2023,ok
7,BRA,676.0,2014-2023,ok
8,BRN,177.0,2014-2023,ok
9,CAF,NaN,None,echec: CAF : aucune donnée annuelle USD trouvé...


## 11. Dataset PIB consolidé (années entières, USD, tous pays)

Un indicateur WEO précis existe déjà pour ça : `NGDPD` = "Gross domestic product (GDP), Current
prices, US dollar"  pas besoin de choisir une devise, c'est la version USD par construction.
On écarte toute période qui n'est pas une année entière à 4 chiffres.

**Format large** (comme `BDP_<CODE>.csv`, label-encodé à l'étape 3) : une ligne par pays, une
colonne par année  pas de colonne `ANNEE` générique.

In [11]:
def construire_dataset_pib():
    """
    Concatène tous les PIB_<CODE>.csv en format LARGE (années en colonnes, comme
    BDP_<CODE>.csv à l'étape 3) : une ligne par pays, une colonne par année.
    Indicateur retenu : NGDPD (PIB en USD courants). FREQUENCY == 'A' uniquement.
    OBS_VALUE est déjà en dollars bruts (comme pour le BOP)  on divise simplement par
    1e6 pour l'exprimer en Millions USD (même convention que BDP_<CODE>.csv). SCALE
    n'est qu'un indicateur d'affichage, pas un facteur à appliquer en plus.
    Retourne (chemin_sortie, dataframe) avec COUNTRY_CODE + une colonne par année.
    """
    fichiers = sorted(RAW_DIR_PIB.glob('PIB_*.csv'))
    lignes = []
    barre = tqdm(fichiers, desc='Consolidation PIB', unit='pays', mininterval=0, miniters=1)
    for chemin in barre:
        code_pays = chemin.stem.replace('PIB_', '')
        barre.set_postfix_str(code_pays, refresh=True)
        pib = pd.read_csv(chemin, low_memory=False)
        f = pib[
            (pib['FREQUENCY'] == FREQUENCE)
            & (pib['INDICATOR'] == 'NGDPD')
            & (pib['TIME_PERIOD'].astype(str).str.fullmatch(r'\d{4}'))
        ].copy()
        if f.empty:
            continue
        f['PIB_USD_MILLIONS'] = pd.to_numeric(f['OBS_VALUE'], errors='coerce') / 1e6

        ligne = {'COUNTRY_CODE': code_pays}
        for _, r in f.iterrows():
            ligne[str(r['TIME_PERIOD'])] = r['PIB_USD_MILLIONS']
        lignes.append(ligne)
    barre.close()

    df = pd.DataFrame(lignes)
    chemin_sortie = BASE / 'data' / 'PIB_CONSOLIDE.csv'
    df.to_csv(chemin_sortie, index=False)
    return chemin_sortie, df


chemin_pib_conso, df_pib_conso = construire_dataset_pib()
print(f"{len(df_pib_conso)} pays -> {chemin_pib_conso}")
print(f"Colonnes années : {[c for c in df_pib_conso.columns if c != 'COUNTRY_CODE']}")
display(df_pib_conso.head(10))

Consolidation PIB: 100%|██████████| 57/57 [00:06<00:00,  8.40pays/s, VNM] 

56 pays -> m:\Travail\BEAC\balance_de_paiement\data\PIB_CONSOLIDE.csv
Colonnes années : ['2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


,COUNTRY_CODE,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,AGO,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.150,119574.114,141728.228
1,ALB,13246.309,11389.178,11862.205,13053.424,15379.684,15581.895,15270.738,17984.503,19188.560,23633.303,27083.523,30279.251
2,ARE,424935.875,381973.043,381717.086,403365.010,440560.109,433926.208,357161.879,422441.389,511403.404,522622.192,552324.847,571643.265
3,ARG,563614.236,642464.260,556774.027,643861.436,524431.466,446762.267,385218.414,486039.550,633520.291,648894.711,637234.361,681485.276
4,AUS,1463091.484,1238862.530,1267659.506,1386366.556,1422597.359,1391987.377,1366669.129,1660123.402,1731233.002,1748174.806,1799417.810,1839960.586
5,AZE,75234.702,52996.784,37829.587,41375.176,47112.941,48174.235,42693.000,54825.412,78807.471,72428.471,74426.000,75937.647
6,BOL,39306.938,40033.529,41605.739,46262.188,48766.911,49414.199,42622.194,48226.857,51330.504,52721.695,55281.337,64249.550
7,BRA,2456054.513,1800045.633,1796621.874,2063518.641,1916933.898,1873286.306,1476091.674,1670650.290,1951849.476,2191137.482,2185822.364,2279917.849
8,BRN,17092.716,12933.379,11403.130,12130.139,13564.418,13472.221,12007.567,14007.959,16684.029,15097.857,15334.492,15941.667
9,CAF,2050.595,1865.526,1994.874,2198.131,2382.769,2444.245,2503.375,2710.741,2564.794,2722.259,2867.888,3191.970


## 12. Dataset BOP consolidé (tous pays, années en colonnes)

Concatène tous les `BDP_<CODE>.csv` **tels quels** (années déjà en colonnes, label-encodées à
l'étape 3  pas de melt, pas de colonne `ANNEE` générique). Ajoute `COUNTRY_CODE` (déduit du nom
de fichier  `COUNTRY` dans `BDP_<CODE>.csv` est le libellé, ex. "Cameroon", pas le code) et les
codes bruts `ENTREE_CODE`/`INDICATEUR_CODE` extraits de `SERIES_CODE` (qui garde les codes même
quand les autres colonnes ont été remplacées par leurs libellés).

In [12]:
def construire_dataset_bop():
    """
    Concatène tous les BDP_<CODE>.csv tels quels  années déjà en colonnes (label
    encodées à l'étape 3), pas de melt, pas de colonne ANNEE générique. Ajoute
    COUNTRY_CODE (déduit du nom de fichier) et ENTREE_CODE / INDICATEUR_CODE (extraits
    de SERIES_CODE, qui garde les codes bruts même si les autres colonnes du fichier
    ont été remplacées par leurs libellés)  nécessaires pour filtrer sur la hiérarchie
    standard en section 14.
    Retourne (chemin_sortie, dataframe).
    """
    fichiers = sorted(DIR_BDP.glob('BDP_*.csv'))
    frames = []
    barre = tqdm(fichiers, desc='Consolidation BOP', unit='pays', mininterval=0, miniters=1)
    for chemin in barre:
        code_pays = chemin.stem.replace('BDP_', '')
        barre.set_postfix_str(code_pays, refresh=True)
        bdp = pd.read_csv(chemin)
        bdp['COUNTRY_CODE'] = code_pays
        frames.append(bdp)
    barre.close()

    df = pd.concat(frames, ignore_index=True)
    parts = df['SERIES_CODE'].str.split('.', expand=True)
    df['ENTREE_CODE'] = parts[1]
    df['INDICATEUR_CODE'] = parts[2]

    chemin_sortie = BASE / 'data' / 'BOP_CONSOLIDE.csv'
    df.to_csv(chemin_sortie, index=False)
    return chemin_sortie, df


chemin_bop_conso, df_bop_conso = construire_dataset_bop()
print(f"{len(df_bop_conso)} séries -> {chemin_bop_conso}")
print(f"{df_bop_conso['COUNTRY_CODE'].nunique()} pays couverts")
display(df_bop_conso.head(10))

Consolidation BOP: 100%|██████████| 47/47 [00:02<00:00, 21.36pays/s, VNM]


29391 séries -> m:\Travail\BEAC\balance_de_paiement\data\BOP_CONSOLIDE.csv
47 pays couverts


,DATASET,SERIES_CODE,OBS_MEASURE,COUNTRY,BOP_ACCOUNTING_ENTRY,INDICATOR,UNIT,FREQUENCY,SCALE,2014,...,2017,2018,2019,2020,2021,2022,2023,COUNTRY_CODE,ENTREE_CODE,INDICATEUR_CODE
0,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D1_F3.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets",Direct investor in direct invesment enterprise...,US dollar,Annual,Millions,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,AGO,A_NFA_T,D1_F3
1,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D1_F5A.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets",Direct investor in direct invesment enterprise...,US dollar,Annual,Millions,1000.105046,...,1352.005057,5.714167,-2349.425835,90.514469,-1057.370588,41.066362,33.356159,AGO,A_NFA_T,D1_F5A
2,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D1_FL.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets",Direct investor in direct invesment enterprise...,US dollar,Annual,Millions,-113.312589,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,AGO,A_NFA_T,D1_FL
3,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D2_F3.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets",Direct investment enterprises in direct invest...,US dollar,Annual,Millions,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,AGO,A_NFA_T,D2_F3
4,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D2_F5A.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets",Direct investment enterprises in direct invest...,US dollar,Annual,Millions,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,AGO,A_NFA_T,D2_F5A
5,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D2_FL.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets",Direct investment enterprises in direct invest...,US dollar,Annual,Millions,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,AGO,A_NFA_T,D2_FL
6,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D3_F3.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets","Direct investment between fellow enterprises, ...",US dollar,Annual,Millions,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,AGO,A_NFA_T,D3_F3
7,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D3_F5A.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets","Direct investment between fellow enterprises, ...",US dollar,Annual,Millions,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,AGO,A_NFA_T,D3_F5A
8,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D3_FL.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets","Direct investment between fellow enterprises, ...",US dollar,Annual,Millions,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,AGO,A_NFA_T,D3_FL
9,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D_F.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets","Direct investment, Total financial assets/liab...",US dollar,Annual,Millions,886.792457,...,1352.005057,5.714167,-2349.425835,90.514469,-1057.370588,41.066362,33.356159,AGO,A_NFA_T,D_F


## 13. Croisement du PIB dans le dataset BOP

Ajoute une colonne `PIB_<année>` pour chaque année **déjà présente dans le BOP consolidé**
(pas toutes les années du PIB  seulement celles qui existent côté BDP, d'où "croisement avec
les années du dataset BDP"), en associant sur `COUNTRY_CODE`. Aucune colonne `ANNEE` générique,
aucune colonne de métadonnées PIB superflue (`DATASET`, `SCALE`...)  juste les valeurs de PIB,
une colonne par année.

In [13]:
def croiser_pib_dans_bop(df_bop, df_pib):
    """
    Ajoute une colonne PIB_<annee> pour chaque année déjà présente dans df_bop (pas
    toutes les années du PIB), en croisant sur COUNTRY_CODE. Pas de format long, pas
    de colonne ANNEE générique : chaque année reste sa propre colonne, exactement
    comme pour le BOP.
    """
    annees_bop = [c for c in df_bop.columns if re.fullmatch(r'\d{4}', c)]
    pib_par_pays = df_pib.set_index('COUNTRY_CODE')

    annees_sans_pib = [a for a in annees_bop if a not in pib_par_pays.columns]
    if annees_sans_pib:
        print(f"ATTENTION : années présentes dans le BOP mais absentes du PIB consolidé "
              f"(colonne PIB_<annee> vide pour ces années) : {annees_sans_pib}")

    for annee in annees_bop:
        if annee in pib_par_pays.columns:
            df_bop[f'PIB_{annee}'] = df_bop['COUNTRY_CODE'].map(pib_par_pays[annee])
        else:
            df_bop[f'PIB_{annee}'] = pd.NA

    return df_bop


df_bop_conso = croiser_pib_dans_bop(df_bop_conso, df_pib_conso)

colonnes_pib = [c for c in df_bop_conso.columns if c.startswith('PIB_')]
taux_couverture = df_bop_conso[colonnes_pib].notna().any(axis=1).mean()
print(f"Lignes avec au moins une valeur de PIB : {taux_couverture:.1%}")

df_bop_conso.to_csv(chemin_bop_conso, index=False)
print(f"{chemin_bop_conso} mis à jour : {len(df_bop_conso)} lignes, {len(colonnes_pib)} colonnes PIB ({colonnes_pib})")
display(df_bop_conso.head(10))

Lignes avec au moins une valeur de PIB : 97.5%
m:\Travail\BEAC\balance_de_paiement\data\BOP_CONSOLIDE.csv mis à jour : 29391 lignes, 10 colonnes PIB (['PIB_2014', 'PIB_2015', 'PIB_2016', 'PIB_2017', 'PIB_2018', 'PIB_2019', 'PIB_2020', 'PIB_2021', 'PIB_2022', 'PIB_2023'])


,DATASET,SERIES_CODE,OBS_MEASURE,COUNTRY,BOP_ACCOUNTING_ENTRY,INDICATOR,UNIT,FREQUENCY,SCALE,2014,...,PIB_2014,PIB_2015,PIB_2016,PIB_2017,PIB_2018,PIB_2019,PIB_2020,PIB_2021,PIB_2022,PIB_2023
0,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D1_F3.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets",Direct investor in direct invesment enterprise...,US dollar,Annual,Millions,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
1,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D1_F5A.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets",Direct investor in direct invesment enterprise...,US dollar,Annual,Millions,1000.105046,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
2,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D1_FL.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets",Direct investor in direct invesment enterprise...,US dollar,Annual,Millions,-113.312589,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
3,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D2_F3.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets",Direct investment enterprises in direct invest...,US dollar,Annual,Millions,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
4,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D2_F5A.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets",Direct investment enterprises in direct invest...,US dollar,Annual,Millions,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
5,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D2_FL.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets",Direct investment enterprises in direct invest...,US dollar,Annual,Millions,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
6,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D3_F3.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets","Direct investment between fellow enterprises, ...",US dollar,Annual,Millions,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
7,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D3_F5A.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets","Direct investment between fellow enterprises, ...",US dollar,Annual,Millions,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
8,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D3_FL.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets","Direct investment between fellow enterprises, ...",US dollar,Annual,Millions,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
9,IMF.STA:BOP(21.0.0),AGO.A_NFA_T.D_F.USD.A,OBS_VALUE,Angola,"Assets, Net acquisition of financial assets","Direct investment, Total financial assets/liab...",US dollar,Annual,Millions,886.792457,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15


## 13bis. Nettoyage des colonnes

Retire les colonnes constantes ou redondantes (`DATASET`, `SERIES_CODE`, `OBS_MEASURE`, `UNIT` 
toujours USD par construction (section 3) , `FREQUENCY`  toujours Annuel , `SCALE`  toujours
Millions). Remplace les libellés `BOP_ACCOUNTING_ENTRY`/`INDICATOR` par leurs codes bruts déjà
présents `ENTREE_CODE`/`INDICATEUR_CODE`  les libellés restent consultables via `LIBELLES_ENTRY`/
`LIBELLES_INDIC` (section 4) au besoin, pas la peine de les dupliquer dans le fichier consolidé.
Réordonne les colonnes années et `PIB_<année>` en ordre croissant.

In [14]:
def nettoyer_bop_consolide(df):
    """
    Retire les colonnes constantes/redondantes et réordonne. Colonnes retirées :
    DATASET, SERIES_CODE, OBS_MEASURE, UNIT, FREQUENCY, SCALE (toutes fixées par
    construction, section 3) ; COUNTRY, BOP_ACCOUNTING_ENTRY et INDICATOR (libellés 
    remplacés par les codes bruts COUNTRY_CODE/ENTREE_CODE/INDICATEUR_CODE déjà
    présents ; les libellés restent consultables via LIBELLES_COUNTRY/LIBELLES_ENTRY/
    LIBELLES_INDIC, section 4, si besoin). Colonnes années et PIB_<année> réordonnées
    en croissant.
    """
    colonnes_a_retirer = [
        'DATASET', 'SERIES_CODE', 'OBS_MEASURE', 'UNIT', 'FREQUENCY', 'SCALE',
        'COUNTRY', 'BOP_ACCOUNTING_ENTRY', 'INDICATOR',
    ]
    df = df.drop(columns=[c for c in colonnes_a_retirer if c in df.columns])

    colonnes_annees = sorted((c for c in df.columns if re.fullmatch(r'\d{4}', c)), key=int)
    colonnes_pib = sorted((c for c in df.columns if re.fullmatch(r'PIB_\d{4}', c)), key=lambda c: int(c.split('_')[1]))
    colonnes_id = [c for c in df.columns if c not in colonnes_annees and c not in colonnes_pib]

    return df[colonnes_id + colonnes_annees + colonnes_pib]


df_bop_conso = nettoyer_bop_consolide(df_bop_conso)
df_bop_conso.to_csv(chemin_bop_conso, index=False)
print(f"{chemin_bop_conso} nettoyé : {len(df_bop_conso)} lignes")
print(f"Colonnes : {list(df_bop_conso.columns)}")
display(df_bop_conso.head(10))

m:\Travail\BEAC\balance_de_paiement\data\BOP_CONSOLIDE.csv nettoyé : 29391 lignes
Colonnes : ['COUNTRY_CODE', 'ENTREE_CODE', 'INDICATEUR_CODE', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', 'PIB_2014', 'PIB_2015', 'PIB_2016', 'PIB_2017', 'PIB_2018', 'PIB_2019', 'PIB_2020', 'PIB_2021', 'PIB_2022', 'PIB_2023']


,COUNTRY_CODE,ENTREE_CODE,INDICATEUR_CODE,2014,2015,2016,2017,2018,2019,2020,...,PIB_2014,PIB_2015,PIB_2016,PIB_2017,PIB_2018,PIB_2019,PIB_2020,PIB_2021,PIB_2022,PIB_2023
0,AGO,A_NFA_T,D1_F3,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
1,AGO,A_NFA_T,D1_F5A,1000.105046,-777.568027,273.005,1352.005057,5.714167,-2349.425835,90.514469,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
2,AGO,A_NFA_T,D1_FL,-113.312589,-7.302190,0.000,0.000000,0.000000,0.000000,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
3,AGO,A_NFA_T,D2_F3,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
4,AGO,A_NFA_T,D2_F5A,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
5,AGO,A_NFA_T,D2_FL,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
6,AGO,A_NFA_T,D3_F3,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
7,AGO,A_NFA_T,D3_F5A,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
8,AGO,A_NFA_T,D3_FL,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15
9,AGO,A_NFA_T,D_F,886.792457,-784.870217,273.005,1352.005057,5.714167,-2349.425835,90.514469,...,164448.325,131661.044,116472.833,141910.639,115456.269,95209.918,66135.869,83022.276,139170.645,113135.15


## 14. Regroupement par type, sous-type et sous-sous-type de compte

Source : `data/RAPPORT_BALANCE_PAIEMENTS_CLASSIFICATION_DEFINITIVE.csv` (979 lignes = tous les
indicateurs BOP, classés `TYPE_DE_COMPTE` → `SOUS_TYPE_1` → `SOUS_TYPE_2` → `SOUS_TYPE_3` →
`SOUS_TYPE_4`)  remplace la hiérarchie API à 13 couples de la section 10, beaucoup plus fine.

On ne garde que `STATUT_INDICATEUR == 'Indicateur principal'` et on exclut
`TYPE_DE_COMPTE == 'Indicateur complémentaire'` (postes hors BOP, sans `BOP_ACCOUNTING_ENTRY`) 
sinon les postes mémo/complémentaires dupliqueraient des montants déjà comptés ailleurs.

Jointure sur `(INDICATEUR_CODE, ENTREE_CODE)` = `(CODE, BOP_ACCOUNTING_ENTRY)` pour le **solde net**
(`VALEUR`) : le fichier fixe une seule écriture comptable par indicateur (`NETCD_T`, `A_NFA_T` ou
`L_NIL_T` selon le compte  jamais `CD_T`/`DB_T` isolément).

**Crédit et débit bruts ajoutés séparément** (`VALEUR_CREDIT`, `VALEUR_DEBIT`) : pour chaque
indicateur retenu par la classification, on va chercher en plus dans le BOP consolidé les lignes
`ENTREE_CODE == 'CD_T'` et `ENTREE_CODE == 'DB_T'` (même `INDICATEUR_CODE`, indépendamment de
l'écriture nette spécifiée par la classification), et on les agrège aux mêmes clés. Ces colonnes
sont vides pour les postes qui n'ont pas de contrepartie crédit/débit dans les données sources
(ex. certains postes du compte financier, où seule la distinction actifs/passifs existe).

Comme les niveaux sont liés par une **relation de somme** (sous-sous-type → sous-type → type),
le tableau produit reste à la granularité la plus fine (tous les niveaux en colonnes) : pour
obtenir le total d'un niveau, il suffit de regrouper à nouveau ce tableau sur les colonnes
voulues et sommer les 3 colonnes de valeur  pas besoin de stocker un tableau séparé par niveau.

In [15]:
def charger_classification():
    """
    Charge data/RAPPORT_BALANCE_PAIEMENTS_CLASSIFICATION_DEFINITIVE.csv et ne garde
    que les indicateurs exploitables pour un regroupement sans double comptage :
    STATUT_INDICATEUR == 'Indicateur principal' (exclut postes memo/complementaires,
    positions, agregats) et TYPE_DE_COMPTE != 'Indicateur complementaire' (exclut les
    ~149 indicateurs hors BOP, sans BOP_ACCOUNTING_ENTRY).

    ATTENTION - voir section 14bis : meme apres ce filtre, le fichier melange encore
    des agregats et leurs propres composantes (ex. indicateur 'G' = "Goods" a cote de
    'G1'/'G1Z'/'G21'/'G22' qui sont ses sous-postes) pour un nombre de codes non
    identifie avec certitude. AGREGATS_CONNUS ci-dessous exclut les cas confirmes par
    recoupement numerique contre CAB (section 14bis), mais NE GARANTIT PAS une somme
    exacte par sous-compte -- verifie : le residu (somme du groupe - CAB reel) grandit
    proportionnellement a la taille de l'economie sur les 40 pays testes (jusqu'a plus
    de 1000 milliards USD pour les Etats-Unis), donc structurel, pas juste du bruit
    residuel. Pour un solde fiable (compte courant, compte de capital), utiliser
    directement les lignes agregees CAB/KAB (comme le fait la section 15), pas la somme
    des sous-comptes de cette section.
    """
    chemin = BASE / 'data' / 'RAPPORT_BALANCE_PAIEMENTS_CLASSIFICATION_DEFINITIVE.csv'
    df = pd.read_csv(chemin)
    df = df[
        (df['STATUT_INDICATEUR'] == 'Indicateur principal')
        & (df['TYPE_DE_COMPTE'] != 'Indicateur complémentaire')
        & df['BOP_ACCOUNTING_ENTRY'].notna()
    ]

    AGREGATS_CONNUS = ['G', 'S', 'IN1', 'IN2', 'GS', 'GSIN1', 'SC', 'D5Z_S13', 'D751_S1W', 'D751_S13', 'G1Z']
    df = df[~df['CODE'].isin(AGREGATS_CONNUS)]

    return df[['CODE', 'BOP_ACCOUNTING_ENTRY', 'TYPE_DE_COMPTE',
               'SOUS_TYPE_1', 'SOUS_TYPE_2', 'SOUS_TYPE_3', 'SOUS_TYPE_4']]


CLASSIFICATION = charger_classification()
print(f"{len(CLASSIFICATION)} indicateurs retenus (sur 979) après filtre STATUT_INDICATEUR/TYPE_DE_COMPTE/agrégats connus.")
display(CLASSIFICATION.head(10))

CLES_GROUPE = ['COUNTRY_CODE', 'ANNEE', 'TYPE_DE_COMPTE', 'SOUS_TYPE_1', 'SOUS_TYPE_2', 'SOUS_TYPE_3', 'SOUS_TYPE_4', 'PIB_USD_MILLIONS']

# Remise en long en memoire seulement (BOP_CONSOLIDE.csv reste large sur disque, sections 11-13bis)
colonnes_annees_bop = sorted((c for c in df_bop_conso.columns if re.fullmatch(r'\d{4}', c)), key=int)
id_vars = [c for c in df_bop_conso.columns if c not in colonnes_annees_bop and not c.startswith('PIB_')]

long_bop = df_bop_conso.melt(
    id_vars=id_vars, value_vars=colonnes_annees_bop, var_name='ANNEE', value_name='VALEUR',
).dropna(subset=['VALEUR'])

long_pib = df_pib_conso.melt(id_vars='COUNTRY_CODE', var_name='ANNEE', value_name='PIB_USD_MILLIONS')
long_bop = long_bop.merge(long_pib, on=['COUNTRY_CODE', 'ANNEE'], how='left')

# Solde net : jointure sur (INDICATEUR_CODE, ENTREE_CODE) = (CODE, BOP_ACCOUNTING_ENTRY) ---
df_comptes = long_bop.merge(
    CLASSIFICATION,
    left_on=['INDICATEUR_CODE', 'ENTREE_CODE'],
    right_on=['CODE', 'BOP_ACCOUNTING_ENTRY'],
    how='inner',
)
resultat = df_comptes.groupby(CLES_GROUPE, as_index=False, dropna=False)['VALEUR'].sum()

# Credit / debit bruts : meme INDICATEUR_CODE, ENTREE_CODE == 'CD_T'/'DB_T' cette fois ---
classification_par_indicateur = CLASSIFICATION.drop(columns=['BOP_ACCOUNTING_ENTRY']).drop_duplicates(subset=['CODE'])

def agreger_entree(code_entree, nom_colonne):
    sous_ensemble = long_bop[long_bop['ENTREE_CODE'] == code_entree]
    jointure = sous_ensemble.merge(classification_par_indicateur, left_on='INDICATEUR_CODE', right_on='CODE', how='inner')
    agg = jointure.groupby(
        ['COUNTRY_CODE', 'ANNEE', 'TYPE_DE_COMPTE', 'SOUS_TYPE_1', 'SOUS_TYPE_2', 'SOUS_TYPE_3', 'SOUS_TYPE_4'],
        as_index=False, dropna=False,
    )['VALEUR'].sum().rename(columns={'VALEUR': nom_colonne})
    return agg


agg_credit = agreger_entree('CD_T', 'VALEUR_CREDIT')
agg_debit = agreger_entree('DB_T', 'VALEUR_DEBIT')

cles_sans_pib = [c for c in CLES_GROUPE if c != 'PIB_USD_MILLIONS']
resultat = resultat.merge(agg_credit, on=cles_sans_pib, how='left')
resultat = resultat.merge(agg_debit, on=cles_sans_pib, how='left')
resultat = resultat.sort_values(['COUNTRY_CODE', 'ANNEE', 'TYPE_DE_COMPTE', 'SOUS_TYPE_1', 'SOUS_TYPE_2'])

chemin_comptes = BASE / 'data' / 'BOP_PAR_COMPTE.csv'
resultat.to_csv(chemin_comptes, index=False)

print(f"{len(resultat)} lignes (pays x année x type x sous-type x sous-sous-type) -> {chemin_comptes}")
print(f"Types de compte retenus : {sorted(resultat['TYPE_DE_COMPTE'].unique())}")
print(f"Lignes avec crédit/débit renseignés : {resultat['VALEUR_CREDIT'].notna().mean():.1%} / {resultat['VALEUR_DEBIT'].notna().mean():.1%}")
print("Aperçu pour le Cameroun :")
display(resultat[resultat['COUNTRY_CODE'] == 'CMR'].head(20))

526 indicateurs retenus (sur 979) après filtre STATUT_INDICATEUR/TYPE_DE_COMPTE/agrégats connus.


,CODE,BOP_ACCOUNTING_ENTRY,TYPE_DE_COMPTE,SOUS_TYPE_1,SOUS_TYPE_2,SOUS_TYPE_3,SOUS_TYPE_4
0,D9_S1W,NETCD_T,Compte de capital,Transferts en capital,Autres transferts en capital,NaN,NaN
1,IN2_S1W,NETCD_T,Compte courant,Revenu secondaire,Autres revenus secondaires,NaN,NaN
2,O_F6,NETCD_T,Compte financier,Autres investissements,"Assurance, pensions et garanties standardisées",NaN,NaN
3,O_F6_S121,NETCD_T,Compte financier,Autres investissements,"Assurance, pensions et garanties standardisées",NaN,NaN
4,O_F6_S13,NETCD_T,Compte financier,Autres investissements,"Assurance, pensions et garanties standardisées",NaN,NaN
5,D9_S13,NETCD_T,Compte de capital,Transferts en capital,Autres transferts en capital,NaN,NaN
6,D99A_S1W,NETCD_T,Compte de capital,Transferts en capital,Remises / annulations de dette,NaN,NaN
7,D9A_S1W,NETCD_T,Compte de capital,Transferts en capital,Remises / annulations de dette,NaN,NaN
8,IN22_S1W,NETCD_T,Compte courant,Revenu secondaire,Transferts personnels / envois de fonds,NaN,NaN
10,IN2_S13,NETCD_T,Compte courant,Revenu secondaire,Autres revenus secondaires,NaN,NaN


11966 lignes (pays x année x type x sous-type x sous-sous-type) -> m:\Travail\BEAC\balance_de_paiement\data\BOP_PAR_COMPTE.csv
Types de compte retenus : ['Compte courant', 'Compte de capital', 'Compte financier', 'Erreurs et omissions']
Lignes avec crédit/débit renseignés : 80.5% / 81.4%
Aperçu pour le Cameroun :


,COUNTRY_CODE,ANNEE,TYPE_DE_COMPTE,SOUS_TYPE_1,SOUS_TYPE_2,SOUS_TYPE_3,SOUS_TYPE_4,PIB_USD_MILLIONS,VALEUR,VALEUR_CREDIT,VALEUR_DEBIT
2825,CMR,2014,Compte courant,Biens,Marchandises générales,NaN,NaN,36395.922,-449.411050,6571.041691,7020.452741
2826,CMR,2014,Compte courant,Revenu primaire,Rémunération des salariés,NaN,NaN,36395.922,55.087790,91.542945,36.455155
2827,CMR,2014,Compte courant,Revenu secondaire,Autres revenus secondaires,NaN,NaN,36395.922,359.326311,1281.657935,281.502656
2828,CMR,2014,Compte courant,Revenu secondaire,Transferts personnels / envois de fonds,NaN,NaN,36395.922,249.029214,513.235925,264.206711
2829,CMR,2014,Compte courant,Services,Services de fabrication sur intrants physiques...,NaN,NaN,36395.922,0.000000,0.000000,NaN
2830,CMR,2014,Compte courant,Services,Services d’assurance,NaN,NaN,36395.922,-74.114869,63.685131,137.800000
2831,CMR,2014,Compte courant,Services,"Services personnels, culturels et récréatifs",NaN,NaN,36395.922,194.427493,200.503353,6.075859
2832,CMR,2014,Compte courant,Services,Transports,Aérien,Passagers,36395.922,32.628052,39.491262,6.863210
2833,CMR,2014,Compte courant,Services,Transports,Maritime,Passagers,36395.922,35.691762,44.059587,8.367825
2834,CMR,2014,Compte courant,Services,"Télécommunications, informatique et information",NaN,NaN,36395.922,66.559012,107.267268,40.708256


---
## 14bis-ter. Diagnostic prealable : repartition des entrees BOP brutes par pays

Avant de fixer le seuil de couverture (section suivante) : pour chaque pays, le nombre d'entrees
BOP brutes (`ENTREE_CODE x INDICATEUR_CODE`, sans tri en compte/sous-compte) qu'il a **en commun
avec au moins un autre pays**, vs celles qui lui sont **propres** (rapportees par personne
d'autre). CEMAC en gras.


In [16]:
CEMAC = {'CMR', 'GAB', 'COG', 'TCD', 'CAF', 'GNQ'}

chemin_bop_conso_diag = BASE / 'data' / 'BOP_CONSOLIDE.csv'
bop_brut_diag = pd.read_csv(chemin_bop_conso_diag, low_memory=False)

annees_bop_diag = [c for c in bop_brut_diag.columns if re.fullmatch(r'\d{4}', c)]
bop_brut_diag['CLE_ENTREE'] = bop_brut_diag['ENTREE_CODE'].astype(str) + '__' + bop_brut_diag['INDICATEUR_CODE'].astype(str)
bop_brut_diag['A_UNE_VALEUR'] = bop_brut_diag[annees_bop_diag].notna().any(axis=1)

presence = bop_brut_diag.loc[bop_brut_diag['A_UNE_VALEUR'], ['COUNTRY_CODE', 'CLE_ENTREE']].drop_duplicates()
entrees_par_pays = presence.groupby('COUNTRY_CODE')['CLE_ENTREE'].apply(set).to_dict()

# Pour chaque pays : entrees partagees avec au moins un autre pays, vs entrees propres (uniques
# a ce pays parmi tous les pays extraits).
compte_commun, compte_propre = {}, {}
for pays, entrees in entrees_par_pays.items():
    autres = set().union(*(s for p2, s in entrees_par_pays.items() if p2 != pays))
    compte_commun[pays] = len(entrees & autres)
    compte_propre[pays] = len(entrees - autres)

ordre = sorted(entrees_par_pays, key=lambda p: compte_commun[p] + compte_propre[p], reverse=True)

fig_diag = go.Figure()
fig_diag.add_trace(go.Bar(x=ordre, y=[compte_commun[p] for p in ordre],
                           name='En commun avec >= 1 autre pays', marker_color='steelblue'))
fig_diag.add_trace(go.Bar(x=ordre, y=[compte_propre[p] for p in ordre],
                           name='Propres a ce pays uniquement', marker_color='indianred'))

ticktext = [f"<b>{p}</b>" if p in CEMAC else p for p in ordre]
fig_diag.update_xaxes(tickvals=ordre, ticktext=ticktext, tickfont=dict(size=9))
fig_diag.update_layout(
    barmode='stack', template='plotly_white', height=520, width=1500,
    title="Repartition des entrees BOP brutes par pays (sans tri par compte) -- CEMAC en gras",
    xaxis_title='Pays', yaxis_title="Nombre d'entrees (ENTREE_CODE x INDICATEUR_CODE)",
)
fig_diag.show()

moyenne_commun = sum(compte_commun.values()) / len(compte_commun)
moyenne_totale = sum(compte_commun[p] + compte_propre[p] for p in ordre) / len(ordre)
print(f"{len(entrees_par_pays)} pays. En moyenne {moyenne_commun:.0f} entrees en commun sur "
      f"{moyenne_totale:.0f} entrees totales par pays.")


# Perimetre retenu pour la suite (section 14ter) : pays avec au moins 700 entrees BOP brutes
# au total (commun + propres) -- exclut les pays a l'extraction trop pauvre/incomplete.
SEUIL_TOTAL_ENTREES = 700
totaux_par_pays = {p: compte_commun[p] + compte_propre[p] for p in entrees_par_pays}
PAYS_RETENUS_700 = sorted(p for p, total in totaux_par_pays.items() if total >= SEUIL_TOTAL_ENTREES)

print(f"{len(PAYS_RETENUS_700)}/{len(entrees_par_pays)} pays retenus (>= {SEUIL_TOTAL_ENTREES} entrees) : "
      f"{PAYS_RETENUS_700}")
pays_exclus_700 = sorted(set(entrees_par_pays) - set(PAYS_RETENUS_700))
if pays_exclus_700:
    print(f"{len(pays_exclus_700)} pays exclus (< {SEUIL_TOTAL_ENTREES} entrees) : {pays_exclus_700}")


47 pays. En moyenne 625 entrees en commun sur 625 entrees totales par pays.
20/47 pays retenus (>= 700 entrees) : ['AGO', 'ALB', 'ARG', 'AZE', 'BOL', 'CMR', 'COG', 'COL', 'ECU', 'GHA', 'ITA', 'KAZ', 'LBY', 'NLD', 'PER', 'ROU', 'RUS', 'SDN', 'TTO', 'VEN']
27 pays exclus (< 700 entrees) : ['ARE', 'AUS', 'BRA', 'BRN', 'CAN', 'CHN', 'DNK', 'DZA', 'EGY', 'GAB', 'GBR', 'GUY', 'IDN', 'IRQ', 'KWT', 'MEX', 'MYS', 'NGA', 'NOR', 'OMN', 'QAT', 'SAU', 'SSD', 'TUN', 'TUR', 'USA', 'VNM']


---
## 14ter. Table clustering : entrees BOP brutes (non classifiees) + indicateurs macro

Pas la version classifiee/agregee de `BOP_PAR_COMPTE.csv` (section 14, triee en compte/sous-
compte) : les entrees BRUTES de `BOP_CONSOLIDE.csv` (couples `ENTREE_CODE x INDICATEUR_CODE`,
avant tout tri), filtrees aux seules entrees presentes pour la majorite des pays (couverture
> 50% -- `SEUIL_COUVERTURE_ENTREE`). Plus les 12 indicateurs macro par pays (section precedente),
et le PIB. Sortie : `DONNEES_CLUSTERING.csv`, une ligne par `(COUNTRY_CODE, ANNEE)`.


In [17]:
INDICATEURS_PAYS = [
    'NGDP_RPCH', 'GGXCNL_NGDP', 'PPPPC', 'PPPGDP', 'GGXONLB_NGDP', 'PCPIPCH',
    'PCPIEPCH', 'GGX_NGDP', 'GGR_NGDP', 'GGXWDG_NGDP', 'NGDPRPPPPC', 'NID_NGDP',
]


def indicateurs_pays_long(code_pays, indicateurs):
    """Lit PIB_<code_pays>.csv (deja en cache) -> DataFrame long (ANNEE en ligne, un
    indicateur par colonne), pret a fusionner sur (COUNTRY_CODE, ANNEE)."""
    chemin = chemin_cache_pib(code_pays)
    if not en_cache(chemin):
        return pd.DataFrame(columns=['COUNTRY_CODE', 'ANNEE'] + indicateurs)

    pib = pd.read_csv(chemin, low_memory=False, dtype={'TIME_PERIOD': str})
    f = pib[
        (pib['FREQUENCY'] == FREQUENCE)
        & pib['INDICATOR'].isin(indicateurs)
        & pib['TIME_PERIOD'].str.fullmatch(r'\d{4}')
    ].copy()
    f['OBS_VALUE'] = pd.to_numeric(f['OBS_VALUE'], errors='coerce')

    large = f.pivot_table(index='TIME_PERIOD', columns='INDICATOR', values='OBS_VALUE', aggfunc='first')
    large = large.reindex(columns=indicateurs)
    large.index = large.index.astype(int)
    large.index.name = 'ANNEE'
    large = large.reset_index()
    large.insert(0, 'COUNTRY_CODE', code_pays)
    return large


df_indicateurs_pays = pd.concat(
    [indicateurs_pays_long(code_pays, INDICATEURS_PAYS) for code_pays in PAYS_RETENUS_700],
    ignore_index=True,
)

chemin_indicateurs_pays = BASE / 'data' / 'INDICATEURS_MACRO_PAYS.csv'
df_indicateurs_pays.to_csv(chemin_indicateurs_pays, index=False)
print(f"{chemin_indicateurs_pays} : {len(df_indicateurs_pays)} lignes "
      f"({df_indicateurs_pays['COUNTRY_CODE'].nunique()} pays), {len(INDICATEURS_PAYS)} indicateurs.")

# Entrees BOP brutes (non classifiees), filtrees a la couverture pays ---
SEUIL_COUVERTURE_ENTREE = 0.50  # gardee si plus de 50% des pays retenus (PAYS_RETENUS_700) la renseignent

chemin_bop_conso = BASE / 'data' / 'BOP_CONSOLIDE.csv'
bop_brut = pd.read_csv(chemin_bop_conso, low_memory=False)
bop_brut = bop_brut[bop_brut['COUNTRY_CODE'].isin(PAYS_RETENUS_700)]

annees_bop = [c for c in bop_brut.columns if re.fullmatch(r'\d{4}', c)]
bop_brut['CLE_ENTREE'] = bop_brut['ENTREE_CODE'].astype(str) + '__' + bop_brut['INDICATEUR_CODE'].astype(str)
bop_brut['A_UNE_VALEUR'] = bop_brut[annees_bop].notna().any(axis=1)

couverture = (
    bop_brut[bop_brut['A_UNE_VALEUR']].groupby('CLE_ENTREE')['COUNTRY_CODE'].nunique() / len(PAYS_RETENUS_700)
)
entrees_retenues = couverture[couverture > SEUIL_COUVERTURE_ENTREE].index.tolist()
print(f"{len(entrees_retenues)}/{bop_brut['CLE_ENTREE'].nunique()} entrees brutes retenues "
      f"(couverture > {SEUIL_COUVERTURE_ENTREE:.0%} des {len(PAYS_RETENUS_700)} pays retenus).")

bop_filtre = bop_brut[bop_brut['CLE_ENTREE'].isin(entrees_retenues)]

# Passage en long (ANNEE en ligne), puis pivot (une colonne par entree brute retenue)
long_entrees = bop_filtre.melt(
    id_vars=['COUNTRY_CODE', 'CLE_ENTREE'], value_vars=annees_bop, var_name='ANNEE', value_name='VALEUR',
).dropna(subset=['VALEUR'])
long_entrees['ANNEE'] = long_entrees['ANNEE'].astype(int)

pivot_entrees = long_entrees.pivot_table(
    index=['COUNTRY_CODE', 'ANNEE'], columns='CLE_ENTREE', values='VALEUR', aggfunc='first',
).reset_index()

# PIB : deja en colonnes PIB_<annee> dans BOP_CONSOLIDE.csv (section 13) -> une colonne
# PIB_USD_MILLIONS par ligne (COUNTRY_CODE, ANNEE).
colonnes_pib = [c for c in bop_brut.columns if re.fullmatch(r'PIB_\d{4}', c)]
pib_long = bop_brut[['COUNTRY_CODE'] + colonnes_pib].drop_duplicates('COUNTRY_CODE').melt(
    id_vars='COUNTRY_CODE', value_vars=colonnes_pib, var_name='ANNEE_COL', value_name='PIB_USD_MILLIONS',
)
pib_long['ANNEE'] = pib_long['ANNEE_COL'].str.replace('PIB_', '', regex=False).astype(int)
pib_long = pib_long.drop(columns='ANNEE_COL').dropna(subset=['PIB_USD_MILLIONS'])

pivot_entrees = pivot_entrees.merge(pib_long, on=['COUNTRY_CODE', 'ANNEE'], how='left')

# Correspondance code -> libelle (documentation, pas necessaire aux calculs -- les codes
# bruts CLE_ENTREE sont deja courts, ex. NETCD_T__CAB) : reutilise LIBELLES_ENTRY/LIBELLES_INDIC
# deja charges en section 4. ---
libelle_entry = LIBELLES_ENTRY.set_index('CODE')['LIBELLE']
libelle_indic = LIBELLES_INDIC.set_index('CODE')['LIBELLE']

correspondance = [
    {
        'CODE_COLONNE': cle,
        'LIBELLE_COMPLET': (
            f"{libelle_entry.get(cle.split('__')[0], cle.split('__')[0])} - "
            f"{libelle_indic.get(cle.split('__')[1], cle.split('__')[1])}"
        ),
    }
    for cle in entrees_retenues
] + [{'CODE_COLONNE': 'PIB_USD_MILLIONS', 'LIBELLE_COMPLET': 'Produit interieur brut, prix courants, US dollar (millions)'}]

df_correspondance = pd.DataFrame(correspondance)
chemin_correspondance = BASE / 'data' / 'CORRESPONDANCE_COLONNES_CLUSTERING.csv'
df_correspondance.to_csv(chemin_correspondance, index=False)
print(f"{chemin_correspondance} : {len(df_correspondance)} colonnes documentees.")

# Fusion des indicateurs macro sur (COUNTRY_CODE, ANNEE) ---
donnees_clustering = pivot_entrees.merge(df_indicateurs_pays, on=['COUNTRY_CODE', 'ANNEE'], how='left')

chemin_clustering = BASE / 'data' / 'DONNEES_CLUSTERING.csv'
donnees_clustering.to_csv(chemin_clustering, index=False)

n_colonnes_features = len(entrees_retenues) + 1 + len(INDICATEURS_PAYS)  # + PIB
print(f"{chemin_clustering} : {len(donnees_clustering)} lignes, {n_colonnes_features} colonnes de features "
      f"({len(entrees_retenues)} entrees BOP brutes + PIB + {len(INDICATEURS_PAYS)} indicateurs macro).")
display(donnees_clustering.head(5))


m:\Travail\BEAC\balance_de_paiement\data\INDICATEURS_MACRO_PAYS.csv : 240 lignes (20 pays), 12 indicateurs.
892/1173 entrees brutes retenues (couverture > 50% des 20 pays retenus).
m:\Travail\BEAC\balance_de_paiement\data\CORRESPONDANCE_COLONNES_CLUSTERING.csv : 893 colonnes documentees.
m:\Travail\BEAC\balance_de_paiement\data\DONNEES_CLUSTERING.csv : 188 lignes, 905 colonnes de features (892 entrees BOP brutes + PIB + 12 indicateurs macro).


,COUNTRY_CODE,ANNEE,A_NFA_T__D1_F3,A_NFA_T__D1_F5A,A_NFA_T__D1_FL,A_NFA_T__D2_F3,A_NFA_T__D2_F5A,A_NFA_T__D2_FL,A_NFA_T__D3_F3,A_NFA_T__D3_F5A,...,PPPPC,PPPGDP,GGXONLB_NGDP,PCPIPCH,PCPIEPCH,GGX_NGDP,GGR_NGDP,GGXWDG_NGDP,NGDPRPPPPC,NID_NGDP
0,AGO,2014,0.0,1000.105046,-113.312589,0.0,0.0,0.0,0.0,0.0,...,8817.904573,2.395011e+11,-4.153251,7.298250,7.497941,32.303123,27.225101,35.274186,11531.905194,28.873478
1,AGO,2015,0.0,-777.568027,-7.302190,0.0,0.0,0.0,0.0,0.0,...,8649.854907,2.435609e+11,-1.010223,9.159489,12.090839,23.873170,21.291085,50.385819,11208.115430,29.108610
2,AGO,2016,0.0,273.005000,0.000000,0.0,0.0,0.0,0.0,0.0,...,8282.040268,2.416954e+11,-1.466382,30.694415,41.125453,19.138242,15.204064,65.691623,10630.503261,29.692073
3,AGO,2017,0.0,1352.005057,0.000000,0.0,0.0,0.0,0.0,0.0,...,8125.706394,2.456794e+11,-2.567522,29.844480,23.666550,20.724753,15.042032,59.607197,10246.338596,28.756030
4,AGO,2018,0.0,5.714167,0.000000,0.0,0.0,0.0,0.0,0.0,...,8370.228757,2.619643e+11,6.083505,19.628661,18.604000,17.830276,19.829072,81.635503,9840.146781,23.934661


---
## 14bis. Vérification (indicative) : somme des sous-comptes vs CAB/KAB réels

**Ceci est un contrôle de cohérence à titre indicatif, pas une source fiable.** On compare, pour
chaque pays et année, la somme des lignes de `resultat` (section 14, hors `Solde du compte
courant`/`Solde du compte de capital` eux-mêmes) au `CAB`/`KAB` réel lu directement dans le BOP
consolidé. Constaté empiriquement sur les 40 pays disponibles : l'écart n'est **pas** du bruit
négligeable  il croît proportionnellement à la taille de l'économie (jusqu'à plus de 1000
milliards USD d'écart pour les États-Unis), signe d'un problème structurel dans le fichier de
classification (agrégats et détails mélangés, cf. `charger_classification()` ci-dessus) qui
dépasse ce qu'une liste d'exclusions manuelle peut corriger de façon fiable.

**Conséquence pratique** : pour un solde réel (compte courant, compte de capital), toujours utiliser
directement les lignes `CAB`/`KAB` (comme le fait la section 15), jamais une resommation des
sous-comptes de cette section.

In [18]:
def verifier_coherence_soldes():
    """
    Compare, pour chaque (pays, année), la somme des sous-comptes retenus par la
    classification (hors les lignes 'Solde du ...' elles-mêmes) au solde réel CAB/KAB
    lu directement dans BOP_CONSOLIDE. Retourne un DataFrame d'écarts, à titre indicatif
    uniquement (voir avertissement ci-dessus).
    """
    detail = resultat[~resultat['SOUS_TYPE_1'].isin(['Solde du compte courant', 'Solde du compte de capital'])]

    sous_total_cc = detail[detail['TYPE_DE_COMPTE'] == 'Compte courant'].groupby(['COUNTRY_CODE', 'ANNEE'])['VALEUR'].sum()
    sous_total_kab = detail[detail['TYPE_DE_COMPTE'] == 'Compte de capital'].groupby(['COUNTRY_CODE', 'ANNEE'])['VALEUR'].sum()

    reel = long_bop[long_bop['INDICATEUR_CODE'].isin(['CAB', 'KAB']) & (long_bop['ENTREE_CODE'] == 'NETCD_T')]
    cab_reel = reel[reel['INDICATEUR_CODE'] == 'CAB'].set_index(['COUNTRY_CODE', 'ANNEE'])['VALEUR']
    kab_reel = reel[reel['INDICATEUR_CODE'] == 'KAB'].set_index(['COUNTRY_CODE', 'ANNEE'])['VALEUR']

    ecarts = pd.DataFrame({
        'SOMME_SOUS_COMPTES_CC': sous_total_cc, 'CAB_REEL': cab_reel,
        'SOMME_SOUS_COMPTES_KAB': sous_total_kab, 'KAB_REEL': kab_reel,
    })
    ecarts['ECART_CC'] = ecarts['SOMME_SOUS_COMPTES_CC'] - ecarts['CAB_REEL']
    ecarts['ECART_KAB'] = ecarts['SOMME_SOUS_COMPTES_KAB'] - ecarts['KAB_REEL']
    return ecarts.reset_index()


ecarts_verification = verifier_coherence_soldes()
resume_ecarts = ecarts_verification.groupby('COUNTRY_CODE')[['ECART_CC', 'ECART_KAB']].agg(['mean', lambda s: s.abs().max()])
print("Écart moyen et max (valeur absolue) par pays, compte courant et compte de capital :")
display(resume_ecarts)

chemin_ecarts = BASE / 'data' / 'VERIFICATION_ECARTS_PAR_COMPTE.csv'
ecarts_verification.to_csv(chemin_ecarts, index=False)
print("Détail par pays/année ->", chemin_ecarts)

Écart moyen et max (valeur absolue) par pays, compte courant et compte de capital :


ECART_CC                    ECART_KAB              
                       mean     <lambda_0>          mean    <lambda_0>
COUNTRY_CODE                                                          
AGO            15930.321304   27871.011811  4.010026e+03  2.351931e+04
ALB              694.993053    1296.728904 -1.448333e+03  2.273905e+03
ARE           -69842.804629   69842.804629  1.402199e+05  1.402199e+05
ARG            16811.333747   24465.420340 -2.333719e+04  6.170164e+04
AUS            32448.342974   82893.280554 -3.600329e+04  1.188321e+05
AZE             6044.554159   10927.936000  1.203867e+04  4.696144e+04
BOL             1317.096750    3787.462862 -1.016651e+03  3.776972e+03
BRA            80829.276442  101824.146921 -9.646141e+04  2.201535e+05
BRN                7.918607    3298.872676  3.270622e+03  6.511200e+03
CAN            -1520.077114   25546.235778 -6.500246e+04  1.095621e+05
CHN           286941.025842  350288.762796  4.408274e+05  8.076629e+05
CMR             2093.925327    2659.861266 -8.732829e+02  1.311894e+03
COG             2283.316002    2991.412393 -2.666933e+02  4.350469e+03
COL            18719.017943   30229.962613           NaN           NaN
DNK           -48208.878959   77978.041383  5.829301e+04  8.960176e+04
DZA            12355.831554   15840.987916 -2.330046e+04  5.407385e+04
ECU             5814.174967    8146.760138  8.436251e+02  5.344225e+03
EGY            30754.449215   44677.299181 -2.575200e+04  4.150230e+04
GAB             1698.391269    1955.513268  1.563718e+02  1.563718e+02
GBR            56316.375310   97734.732216 -2.253173e+05  3.231058e+05
GHA             4121.991612    8003.509195 -2.850685e+03  4.292564e+03
GUY             1807.641597    6315.408879  1.667105e+02  4.397254e+03
IDN            43109.144924   57426.447524 -2.536298e+04  6.078047e+04
IRQ            12145.460400   14374.300000  3.692228e+04  1.158840e+05
ITA           -10608.347506   22175.072484  9.764736e+04  1.594975e+05
KAZ            21114.869201   25226.503651 -7.894586e+03  2.091167e+04
KWT           -19879.186018   38006.509688  6.232112e+04  1.288567e+05
LBY             2480.550000    6418.000000  5.736000e+03  9.607400e+03
MEX            75005.191290  108666.968035 -2.718450e+04  6.327467e+04
MYS            12493.876165   23622.401020  2.113395e+04  3.000293e+04
NGA            50038.379269   70803.351000           NaN           NaN
NLD           -43181.750883   64022.267837  1.593668e+05  4.753986e+05
NOR           -23599.579346   37965.597903  1.004731e+05  3.422494e+05
OMN             1806.917312    7832.333825 -9.536251e+03  2.467916e+04
PER            10148.046200   18298.682983 -8.384864e+03  2.012075e+04
QAT             1809.532967    8236.813187  3.812616e+04  1.251258e+05
ROU            12788.012817   20151.022677 -3.332144e+03  2.059227e+04
RUS            58931.195496   92104.500000  1.381819e+05  4.617306e+05
SAU            12813.303171   29542.420568  5.436272e+04  2.885214e+05
SDN             1165.993890    2411.211683 -3.811105e+03  5.266850e+03
SSD             2085.979511    3797.070000  3.556089e+02  2.342517e+03
TTO             2282.056466    6165.217479  3.099876e+03  8.309006e+03
TUN             2534.052761    3916.229313 -2.458383e+03  4.134519e+03
TUR           -18831.300000   30363.000000 -4.747260e+04  9.344100e+04
USA          -449878.600000  545416.000000 -1.135361e+06  1.945388e+06
VEN            11548.000000   12618.000000 -1.202733e+04  4.352400e+04
VNM            14766.127600   27996.000000           NaN           NaN

Détail par pays/année -> m:\Travail\BEAC\balance_de_paiement\data\VERIFICATION_ECARTS_PAR_COMPTE.csv


---
## 15. Métriques d'analyse externe

10 métriques calculées par (pays, année) à partir de `BOP_PAR_COMPTE.csv` (section 14) et
`PIB_CONSOLIDE.csv` (section 11). Formules retenues :

| Métrique | Formule |
|---|---|
| TC  Taux de couverture des biens | Crédit(Biens) / Débit(Biens) × 100 |
| TM  Taux de couverture des importations | Débit(Biens) / Crédit(Biens) × 100 |
| TAS  Taux d'absorption des services | \|min(Crédit(S) − Débit(S), 0)\| / Crédit(Biens) × 100 |
| TFR  Taux de fuite de revenu | \|min(Crédit(RP) − Débit(RP), 0)\| / Crédit(Biens) × 100 |
| TRE  Taux de rétention | 100 − TM − TAS − TFR |
| Ratio IDE / Solde courant | IDE net / \|Solde courant\| × 100 |
| Ratio E&O | \|Erreurs et omissions nettes\| / PIB × 100 |
| Solde courant / PIB | Solde courant / PIB × 100 |
| Balance de base | Solde courant + IDE net + Portefeuille long terme net |
| Taux d'ouverture commerciale | (Crédit Biens + Crédit Services + Débit Biens + Débit Services) / PIB × 100 |

**Hypothèses à connaitre** :
- TC/TM/TAS/TFR utilisent `Crédit(Biens)` (recettes d'exportation de biens) comme dénominateur
  commun (sauf TM, son symétrique exact de TC).
- TAS et TFR ne comptent que le déficit NET (services ou revenu primaire importés nets, pas le
  brut) : `\|min(Crédit − Débit, 0)\|` équivaut à `max(Débit − Crédit, 0)` -- un pays exportateur
  net de services ou de revenu primaire a TAS/TFR = 0 (pas de valeur négative), il n'y a donc pas
  de fuite à soustraire de TRE dans ce cas.
- TRE intègre maintenant TM en plus de TAS et TFR : 100 − TM − TAS − TFR.
- "Solde courant"/"Solde du compte de capital" viennent directement des lignes agrégées `CAB`/`KAB`
  de la classification (`SOUS_TYPE_1 == 'Solde du compte courant'`/`'Solde du compte de capital'`),
  pas d'une resommation de leurs composantes (plus sûr, évite tout écart d'arrondi/complétude).
- IDE net et solde courant sont pris tels quels depuis `VALEUR` (net déjà signé par le FMI,
  `A_NFA_T`/`L_NIL_T`/`NETCD_T` selon l'indicateur)  on fait confiance à la convention de signe du
  FMI plutôt que de la recalculer nous-mêmes.
- Portefeuille long terme (balance de base) : titres de dette avec `MATURITE == 'Long terme'`, plus
  les participations/parts de fonds (`MATURITE` vide  pas de notion de maturité pour de l'equity,
  donc traitées comme long terme par convention historique). Titres de dette court terme exclus.


In [19]:
chemin_par_compte = BASE / 'data' / 'BOP_PAR_COMPTE.csv'
par_compte = pd.read_csv(chemin_par_compte)

# Rollup au niveau SOUS_TYPE_1 (les lignes de BOP_PAR_COMPTE sont au niveau SOUS_TYPE_1..4)
niveau1 = par_compte.groupby(
    ['COUNTRY_CODE', 'ANNEE', 'TYPE_DE_COMPTE', 'SOUS_TYPE_1'], as_index=False,
)[['VALEUR', 'VALEUR_CREDIT', 'VALEUR_DEBIT']].sum()


def extraire(type_compte, sous_type1, colonne='VALEUR'):
    """Série (COUNTRY_CODE, ANNEE) -> valeur, pour un (TYPE_DE_COMPTE, SOUS_TYPE_1) donné."""
    sub = niveau1[(niveau1['TYPE_DE_COMPTE'] == type_compte) & (niveau1['SOUS_TYPE_1'] == sous_type1)]
    return sub.set_index(['COUNTRY_CODE', 'ANNEE'])[colonne]


credit_biens     = extraire('Compte courant', 'Biens', 'VALEUR_CREDIT')
debit_biens      = extraire('Compte courant', 'Biens', 'VALEUR_DEBIT')
credit_services   = extraire('Compte courant', 'Services', 'VALEUR_CREDIT')
debit_services    = extraire('Compte courant', 'Services', 'VALEUR_DEBIT')
debit_revenu_primaire = extraire('Compte courant', 'Revenu primaire', 'VALEUR_DEBIT')
credit_revenu_primaire = extraire('Compte courant', 'Revenu primaire', 'VALEUR_CREDIT')
solde_courant     = extraire('Compte courant', 'Solde du compte courant', 'VALEUR')
ide_net           = extraire('Compte financier', 'Investissements directs', 'VALEUR')
eo_net            = extraire('Erreurs et omissions', 'Erreurs et omissions nettes', 'VALEUR')

pib_long = df_pib_conso.melt(id_vars='COUNTRY_CODE', var_name='ANNEE', value_name='PIB_USD_MILLIONS')
pib_long['ANNEE'] = pib_long['ANNEE'].astype(int)  # colonnes de PIB_CONSOLIDE.csv = chaines, ANNEE de BOP_PAR_COMPTE = int64 : sans ce cast, la jointure par (COUNTRY_CODE, ANNEE) ne matchait jamais et produisait deux lignes par annee (PIB seul / reste seul)
pib = pib_long.set_index(['COUNTRY_CODE', 'ANNEE'])['PIB_USD_MILLIONS']

metriques = pd.DataFrame({
    'CREDIT_BIENS': credit_biens, 'DEBIT_BIENS': debit_biens,
    'CREDIT_SERVICES': credit_services, 'DEBIT_SERVICES': debit_services,
    'DEBIT_REVENU_PRIMAIRE': debit_revenu_primaire, 'CREDIT_REVENU_PRIMAIRE': credit_revenu_primaire,
    'SOLDE_COURANT': solde_courant, 'IDE_NET': ide_net, 'EO_NET': eo_net,
    'PIB_USD_MILLIONS': pib,
}).reset_index()

metriques['TC_TAUX_COUVERTURE_BIENS'] = metriques['CREDIT_BIENS'] / metriques['DEBIT_BIENS'] * 100
metriques['TM_TAUX_COUVERTURE_IMPORTATIONS'] = metriques['DEBIT_BIENS'] / metriques['CREDIT_BIENS'] * 100

# TAS/TFR ne comptent que le DEFICIT net (services/revenu primaire importes nets, pas
# le brut) : un surplus net de services ou de revenu primaire n'entre pas en fuite, il
# n'est donc pas soustrait de TRE -- d'ou le clip a 0 (equivalent a |min(credit-debit, 0)|).
metriques['TAS_TAUX_ABSORPTION_SERVICES'] = (metriques['DEBIT_SERVICES'] - metriques['CREDIT_SERVICES']).clip(lower=0) / metriques['CREDIT_BIENS'] * 100
metriques['TFR_TAUX_FUITE_REVENU'] = (metriques['DEBIT_REVENU_PRIMAIRE'] - metriques['CREDIT_REVENU_PRIMAIRE']).clip(lower=0) / metriques['CREDIT_BIENS'] * 100
metriques['TRE_TAUX_RETENTION'] = 100 - metriques['TM_TAUX_COUVERTURE_IMPORTATIONS'] - metriques['TAS_TAUX_ABSORPTION_SERVICES'] - metriques['TFR_TAUX_FUITE_REVENU']
metriques['RATIO_IDE_SOLDE_COURANT'] = metriques['IDE_NET'] / metriques['SOLDE_COURANT'].abs() * 100
metriques['RATIO_ERREURS_OMISSIONS'] = metriques['EO_NET'].abs() / metriques['PIB_USD_MILLIONS'] * 100
metriques['SOLDE_COURANT_PIB'] = metriques['SOLDE_COURANT'] / metriques['PIB_USD_MILLIONS'] * 100
metriques['TAUX_OUVERTURE_COMMERCIALE'] = (metriques['CREDIT_BIENS'] + metriques['CREDIT_SERVICES'] + metriques['DEBIT_BIENS'] + metriques['DEBIT_SERVICES']) / metriques['PIB_USD_MILLIONS'] * 100

print(f"{len(metriques)} lignes (pays x année)  aperçu des 8 premières métriques :")
display(metriques[metriques['COUNTRY_CODE'] == 'CMR'].head(10))

675 lignes (pays x année)  aperçu des 8 premières métriques :


,COUNTRY_CODE,ANNEE,CREDIT_BIENS,DEBIT_BIENS,CREDIT_SERVICES,DEBIT_SERVICES,DEBIT_REVENU_PRIMAIRE,CREDIT_REVENU_PRIMAIRE,SOLDE_COURANT,IDE_NET,...,PIB_USD_MILLIONS,TC_TAUX_COUVERTURE_BIENS,TM_TAUX_COUVERTURE_IMPORTATIONS,TAS_TAUX_ABSORPTION_SERVICES,TFR_TAUX_FUITE_REVENU,TRE_TAUX_RETENTION,RATIO_IDE_SOLDE_COURANT,RATIO_ERREURS_OMISSIONS,SOLDE_COURANT_PIB,TAUX_OUVERTURE_COMMERCIALE
144,CMR,2014,6571.041691,7020.452741,455.213180,207.815033,36.455155,91.542945,-1401.974732,-476.355460,...,36395.922,93.598546,106.839266,0.000000,0.0,-6.839266,-33.977464,0.462763,-3.852010,39.165164
145,CMR,2015,5219.111886,5591.229015,387.857684,187.595565,33.659686,78.482886,-1173.691255,37.448515,...,32213.494,93.344627,107.129894,0.000000,0.0,-7.129894,3.190661,0.068075,-3.643477,35.344797
146,CMR,2016,4566.075038,4779.744902,436.200390,198.241917,56.950352,173.016239,-1037.444249,-64.403595,...,33805.006,95.529681,104.679508,0.000000,0.0,-4.679508,-6.207909,0.006698,-3.068907,29.523031
147,CMR,2017,4603.445015,4830.200668,462.362475,229.455862,36.953899,165.366351,-949.763043,-51.788815,...,36085.808,95.305461,104.925782,0.000000,0.0,-4.925782,-5.452814,0.896544,-2.631957,28.059408
148,CMR,2018,5172.329396,5715.390980,499.733830,274.997830,43.585809,177.467531,-1409.334284,-280.584072,...,39991.969,90.498260,110.499362,0.000000,0.0,-10.499362,-19.908979,0.445644,-3.524043,29.161985
149,CMR,2019,5514.129012,6260.515484,463.897872,284.232309,41.685064,190.846218,-1695.070703,-191.306515,...,39672.660,88.077875,113.535891,0.000000,0.0,-13.535891,-11.286049,0.189805,-4.272642,31.565251
150,CMR,2020,4415.702637,5092.275762,390.093077,272.356421,46.750036,234.130086,-1512.407004,-631.915118,...,40832.221,86.713737,115.321981,0.000000,0.0,-15.321981,-41.782081,1.051091,-3.703955,24.907849
151,CMR,2021,5864.605693,6572.817385,217.569876,261.592276,39.139677,227.145410,-1794.494947,-25.501156,...,45035.591,89.225143,112.076033,0.750646,0.0,-12.826678,-1.421077,0.916327,-3.984615,28.680839
152,CMR,2022,6951.464405,7216.906098,262.242903,346.403033,27.804002,184.819600,-1505.133036,-968.281220,...,44442.500,96.321946,103.818500,1.210682,0.0,-5.029182,-64.331936,0.298906,-3.386697,33.249742
153,CMR,2023,6340.243856,7748.356639,378.931889,504.852778,62.874067,188.198793,-2018.870650,-697.828903,...,48821.693,81.826949,122.209127,1.986058,0.0,-24.195184,-34.565310,0.515294,-4.135192,30.667485


In [20]:
# Balance de base : nécessite MATURITE, absente de BOP_PAR_COMPTE.csv (niveau SOUS_TYPE) ---
# On refait une jointure ciblée sur le brut consolidé pour isoler le portefeuille long terme.
chemin_classification = BASE / 'data' / 'RAPPORT_BALANCE_PAIEMENTS_CLASSIFICATION_DEFINITIVE.csv'
classif_maturite = pd.read_csv(chemin_classification)
classif_maturite = classif_maturite[
    (classif_maturite['STATUT_INDICATEUR'] == 'Indicateur principal')
    & (classif_maturite['SOUS_TYPE_1'] == 'Investissements de portefeuille')
    & classif_maturite['BOP_ACCOUNTING_ENTRY'].notna()
    & (classif_maturite['MATURITE'].isna() | (classif_maturite['MATURITE'] == 'Long terme'))  # garde Long terme + vide (equity, pas de maturité) ; exclut explicitement Court terme
][['CODE', 'BOP_ACCOUNTING_ENTRY']]

portefeuille_lt = long_bop.merge(
    classif_maturite, left_on=['INDICATEUR_CODE', 'ENTREE_CODE'],
    right_on=['CODE', 'BOP_ACCOUNTING_ENTRY'], how='inner',
)
portefeuille_lt_net = portefeuille_lt.groupby(['COUNTRY_CODE', 'ANNEE'])['VALEUR'].sum()

metriques = metriques.set_index(['COUNTRY_CODE', 'ANNEE'])
metriques['PORTEFEUILLE_LT_NET'] = portefeuille_lt_net
metriques['BALANCE_DE_BASE'] = metriques['SOLDE_COURANT'] + metriques['IDE_NET'] + metriques['PORTEFEUILLE_LT_NET'].fillna(0)
metriques = metriques.reset_index()

chemin_metriques = BASE / 'data' / 'METRIQUES_ANALYSE.csv'
metriques.to_csv(chemin_metriques, index=False)

print(f"{chemin_metriques} : {len(metriques)} lignes")
print(f"Colonnes : {list(metriques.columns)}")
print("\nAperçu complet pour le Cameroun :")
display(metriques[metriques['COUNTRY_CODE'] == 'CMR'])

m:\Travail\BEAC\balance_de_paiement\data\METRIQUES_ANALYSE.csv : 675 lignes
Colonnes : ['COUNTRY_CODE', 'ANNEE', 'CREDIT_BIENS', 'DEBIT_BIENS', 'CREDIT_SERVICES', 'DEBIT_SERVICES', 'DEBIT_REVENU_PRIMAIRE', 'CREDIT_REVENU_PRIMAIRE', 'SOLDE_COURANT', 'IDE_NET', 'EO_NET', 'PIB_USD_MILLIONS', 'TC_TAUX_COUVERTURE_BIENS', 'TM_TAUX_COUVERTURE_IMPORTATIONS', 'TAS_TAUX_ABSORPTION_SERVICES', 'TFR_TAUX_FUITE_REVENU', 'TRE_TAUX_RETENTION', 'RATIO_IDE_SOLDE_COURANT', 'RATIO_ERREURS_OMISSIONS', 'SOLDE_COURANT_PIB', 'TAUX_OUVERTURE_COMMERCIALE', 'PORTEFEUILLE_LT_NET', 'BALANCE_DE_BASE']

Aperçu complet pour le Cameroun :


,COUNTRY_CODE,ANNEE,CREDIT_BIENS,DEBIT_BIENS,CREDIT_SERVICES,DEBIT_SERVICES,DEBIT_REVENU_PRIMAIRE,CREDIT_REVENU_PRIMAIRE,SOLDE_COURANT,IDE_NET,...,TM_TAUX_COUVERTURE_IMPORTATIONS,TAS_TAUX_ABSORPTION_SERVICES,TFR_TAUX_FUITE_REVENU,TRE_TAUX_RETENTION,RATIO_IDE_SOLDE_COURANT,RATIO_ERREURS_OMISSIONS,SOLDE_COURANT_PIB,TAUX_OUVERTURE_COMMERCIALE,PORTEFEUILLE_LT_NET,BALANCE_DE_BASE
144,CMR,2014,6571.041691,7020.452741,455.213180,207.815033,36.455155,91.542945,-1401.974732,-476.355460,...,106.839266,0.000000,0.0,-6.839266,-33.977464,0.462763,-3.852010,39.165164,NaN,-1878.330192
145,CMR,2015,5219.111886,5591.229015,387.857684,187.595565,33.659686,78.482886,-1173.691255,37.448515,...,107.129894,0.000000,0.0,-7.129894,3.190661,0.068075,-3.643477,35.344797,NaN,-1136.242741
146,CMR,2016,4566.075038,4779.744902,436.200390,198.241917,56.950352,173.016239,-1037.444249,-64.403595,...,104.679508,0.000000,0.0,-4.679508,-6.207909,0.006698,-3.068907,29.523031,NaN,-1101.847844
147,CMR,2017,4603.445015,4830.200668,462.362475,229.455862,36.953899,165.366351,-949.763043,-51.788815,...,104.925782,0.000000,0.0,-4.925782,-5.452814,0.896544,-2.631957,28.059408,NaN,-1001.551858
148,CMR,2018,5172.329396,5715.390980,499.733830,274.997830,43.585809,177.467531,-1409.334284,-280.584072,...,110.499362,0.000000,0.0,-10.499362,-19.908979,0.445644,-3.524043,29.161985,NaN,-1689.918355
149,CMR,2019,5514.129012,6260.515484,463.897872,284.232309,41.685064,190.846218,-1695.070703,-191.306515,...,113.535891,0.000000,0.0,-13.535891,-11.286049,0.189805,-4.272642,31.565251,NaN,-1886.377219
150,CMR,2020,4415.702637,5092.275762,390.093077,272.356421,46.750036,234.130086,-1512.407004,-631.915118,...,115.321981,0.000000,0.0,-15.321981,-41.782081,1.051091,-3.703955,24.907849,NaN,-2144.322122
151,CMR,2021,5864.605693,6572.817385,217.569876,261.592276,39.139677,227.145410,-1794.494947,-25.501156,...,112.076033,0.750646,0.0,-12.826678,-1.421077,0.916327,-3.984615,28.680839,NaN,-1819.996102
152,CMR,2022,6951.464405,7216.906098,262.242903,346.403033,27.804002,184.819600,-1505.133036,-968.281220,...,103.818500,1.210682,0.0,-5.029182,-64.331936,0.298906,-3.386697,33.249742,NaN,-2473.414255
153,CMR,2023,6340.243856,7748.356639,378.931889,504.852778,62.874067,188.198793,-2018.870650,-697.828903,...,122.209127,1.986058,0.0,-24.195184,-34.565310,0.515294,-4.135192,30.667485,NaN,-2716.699553


---
# Etape 4 : tableaux comparatifs -- tous pays de l'etude

#### 16. Tableaux comparatifs -- tous les pays ayant au moins 10 annees communes avec le Cameroun

Le perimetre n'est plus limite aux 6 pays de la CEMAC : tout pays de `PAYS` (section 1) dont
les donnees classifiees couvrent au moins 10 des annees reellement disponibles pour le
Cameroun entre dans l'etude. Les annees utilisees pour chaque pays sont celles du Cameroun
(pas les siennes propres) -- un pays avec de bonnes donnees sur une autre periode calendaire
reste exclu tant que la fenetre d'extraction (section 1) n'est pas realignee sur lui.


In [21]:
chemin_par_compte_etude = BASE / 'data' / 'BOP_PAR_COMPTE.csv'
chemin_metriques_etude = BASE / 'data' / 'METRIQUES_ANALYSE.csv'

par_compte_all = pd.read_csv(chemin_par_compte_etude)
metriques_all = pd.read_csv(chemin_metriques_etude)

if 'CMR' not in par_compte_all['COUNTRY_CODE'].unique():
    raise RuntimeError("Cameroun (CMR) absent de BOP_PAR_COMPTE.csv -- impossible de definir "
                        "les annees de reference de l'etude.")

# Annees de reference = celles reellement disponibles pour le Cameroun. Tous les autres
# pays sont evalues par rapport a CE MEME jeu d'annees (pas les leurs propres) : un pays
# n'entre dans l'etude que s'il a des donnees classifiees sur au moins 10 de ces annees-la.
# Si un pays a de bonnes donnees mais sur une periode calendaire differente de celle du
# Cameroun, il restera exclu ici -- il faudrait alors elargir la fenetre d'extraction
# (ANNEE_COURANTE / N_ANNEES / MARGE_ANNEES, section 1) et refaire tourner les sections
# 7 a 14 pour que son propre determiner_periode() couvre la meme periode calendaire.
ANNEES_REFERENCE = sorted(par_compte_all[par_compte_all['COUNTRY_CODE'] == 'CMR']['ANNEE'].unique())
# Seuil = toutes les annees de reference disponibles pour le Cameroun (pas un nombre
# fixe) : reste coherent quelle que soit la largeur de la plage d'etude retenue
# (ANNEE_ETUDE_DEBUT/ANNEE_ETUDE_FIN, section 1) -- un seuil fixe de 10 n'aurait plus de
# sens si la plage d'etude est reduite a moins de 10 annees.
SEUIL_ANNEES = len(ANNEES_REFERENCE)

annees_par_pays_brut = {
    code: set(par_compte_all[par_compte_all['COUNTRY_CODE'] == code]['ANNEE'].unique())
    for code in par_compte_all['COUNTRY_CODE'].unique()
}

PAYS_ELIGIBLES = {
    code: nom for code, nom in PAYS.items()
    if len(annees_par_pays_brut.get(code, set()) & set(ANNEES_REFERENCE)) >= SEUIL_ANNEES
}
PAYS_EXCLUS = {code: nom for code, nom in PAYS.items() if code not in PAYS_ELIGIBLES}

if 'CMR' not in PAYS_ELIGIBLES:
    raise RuntimeError(f"Le Cameroun lui-meme n'a que {len(ANNEES_REFERENCE)} annee(s) "
                        f"(< {SEUIL_ANNEES}) -- pas assez pour servir de reference.")

par_compte_etude = par_compte_all[par_compte_all['COUNTRY_CODE'].isin(PAYS_ELIGIBLES)]
metriques_etude = metriques_all[metriques_all['COUNTRY_CODE'].isin(PAYS_ELIGIBLES)]

# Annees par pays restreintes aux annees de reference (Cameroun) uniquement -- pas les
# annees propres du pays hors de cette fenetre, meme s'il en a.
ANNEES_PAR_PAYS = {
    code: sorted(annees_par_pays_brut[code] & set(ANNEES_REFERENCE))
    for code in PAYS_ELIGIBLES
}
ANNEES_UNION = sorted(set.union(*(set(v) for v in ANNEES_PAR_PAYS.values())))

print(f"Annees de reference (Cameroun) : {ANNEES_REFERENCE}")
print(f"Pays eligibles ({len(PAYS_ELIGIBLES)}/{len(PAYS)}, >= {SEUIL_ANNEES} annees communes avec le Cameroun) :")
for code, nom in PAYS_ELIGIBLES.items():
    print(f"  {nom:<28} ({code}) : {ANNEES_PAR_PAYS[code]}")
if PAYS_EXCLUS:
    print(f"\n{len(PAYS_EXCLUS)} pays exclus (< {SEUIL_ANNEES} annees communes avec le Cameroun, "
          f"donnees insuffisantes ou fenetre d'extraction non alignee) :")
    print(', '.join(f"{nom} ({c})" for c, nom in PAYS_EXCLUS.items()))


Annees de reference (Cameroun) : [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Pays eligibles (42/51, >= 10 annees communes avec le Cameroun) :
  Arabie saoudite              (SAU) : [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
  Russie                       (RUS) : [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
  États-Unis                   (USA) : [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
  Canada                       (CAN) : [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
  Irak                         (IRQ) : [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
  Norvège                      (NOR) : [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
  Brésil                       (BRA) : [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
  Kazakhstan                   (KAZ) : [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
  Nigeria                      (NGA) : [2

In [22]:
def extraire_bdp_cemac(code_pays, annee):
    """
    Construit un dict de soldes principaux pour un pays/année, à partir de
    BOP_PAR_COMPTE.csv (agrégé) et METRIQUES_ANALYSE.csv (PIB, crédits/débits biens).
    """
    sub = par_compte_etude[(par_compte_etude['COUNTRY_CODE'] == code_pays) & (par_compte_etude['ANNEE'] == annee)]

    def valeur(type_compte, sous_type1):
        s = sub[(sub['TYPE_DE_COMPTE'] == type_compte) & (sub['SOUS_TYPE_1'] == sous_type1)]['VALEUR']
        return s.sum() if len(s) else 0.0

    solde_courant = valeur('Compte courant', 'Solde du compte courant')
    compte_capital = valeur('Compte de capital', 'Solde du compte de capital')

    # Compte financier net = FABXRRI + RUE (SOUS_TYPE_1 'Autres postes du compte financier') --
    # pas la somme de tous les SOUS_TYPE_1 du compte financier, qui redonderait avec ce total.
    compte_financier = valeur('Compte financier', 'Autres postes du compte financier')

    erreurs_omissions = valeur('Erreurs et omissions', 'Erreurs et omissions nettes')

    # Biens/Services/Revenu primaire/secondaire : sommes de détail, fiabilité limitée (cf. section 14bis)
    detail_cc = sub[(sub['TYPE_DE_COMPTE'] == 'Compte courant') & (sub['SOUS_TYPE_1'] != 'Solde du compte courant')]
    biens = detail_cc[detail_cc['SOUS_TYPE_1'] == 'Biens']['VALEUR'].sum()
    services = detail_cc[detail_cc['SOUS_TYPE_1'] == 'Services']['VALEUR'].sum()
    rev_primaire = detail_cc[detail_cc['SOUS_TYPE_1'] == 'Revenu primaire']['VALEUR'].sum()
    rev_secondaire = detail_cc[detail_cc['SOUS_TYPE_1'] == 'Revenu secondaire']['VALEUR'].sum()

    ligne_metrique = metriques_etude[(metriques_etude['COUNTRY_CODE'] == code_pays) & (metriques_etude['ANNEE'] == annee)]
    exp_biens = ligne_metrique['CREDIT_BIENS'].iloc[0] if len(ligne_metrique) else None
    imp_biens = ligne_metrique['DEBIT_BIENS'].iloc[0] if len(ligne_metrique) else None
    pib = ligne_metrique['PIB_USD_MILLIONS'].iloc[0] if len(ligne_metrique) else None

    return {
        'solde_courant': solde_courant,
        'biens': biens,
        'services': services,
        'rev_primaire': rev_primaire,
        'rev_secondaire': rev_secondaire,
        'compte_capital': compte_capital,
        'capacite_finan': solde_courant + compte_capital,
        'compte_financier': compte_financier,
        'erreurs_omissions': erreurs_omissions,
        'exp_biens': exp_biens,
        'imp_biens': imp_biens,
        'pib': pib,
    }


# BDP_PAR_PAYS[code][annee] -- uniquement pour les (pays, année) réellement disponibles
# (cf. ANNEES_PAR_PAYS, section 16) : pas de clé fabriquée à 0 pour une combinaison absente.
BDP_PAR_PAYS = {
    code: {annee: extraire_bdp_cemac(code, annee) for annee in ANNEES_PAR_PAYS[code]}
    for code in PAYS_ELIGIBLES
}


def bdp_pour(code, annee):
    """None si ce pays n'a pas de données pour cette année (pas de valeur fabriquée)."""
    return BDP_PAR_PAYS.get(code, {}).get(annee)


print(f"Extraction terminée pour {sum(len(v) for v in ANNEES_PAR_PAYS.values())} couples (pays, année).")


Extraction terminée pour 420 couples (pays, année).


In [23]:
def indicateurs_cemac(code_pays, annee):
    CLES_VIDES = ['tc', 'tm', 'tas', 'tfr', 'tre', 'ratio_pib', 'ratio_biens_pib',
                  'ratio_ide_solde_courant', 'ratio_eo_pib', 'taux_ouverture',
                  'ide_net', 'portefeuille_lt_net', 'balance_de_base', 'pib_usd_millions']
    ligne = metriques_etude[(metriques_etude['COUNTRY_CODE'] == code_pays) & (metriques_etude['ANNEE'] == annee)]
    if ligne.empty:
        return {k: None for k in CLES_VIDES}
    r = ligne.iloc[0]
    b = bdp_pour(code_pays, annee)
    pib = r['PIB_USD_MILLIONS']
    ratio_biens_pib = (b['biens'] / pib * 100) if (b is not None and pib) else None

    def arrondi(cle, decimales=1):
        return round(r[cle], decimales) if pd.notna(r[cle]) else None

    return {
        'tc': arrondi('TC_TAUX_COUVERTURE_BIENS', 2),
        'tm': arrondi('TM_TAUX_COUVERTURE_IMPORTATIONS', 2),
        'tas': arrondi('TAS_TAUX_ABSORPTION_SERVICES'),
        'tfr': arrondi('TFR_TAUX_FUITE_REVENU'),
        'tre': arrondi('TRE_TAUX_RETENTION'),
        'ratio_pib': arrondi('SOLDE_COURANT_PIB'),
        'ratio_biens_pib': round(ratio_biens_pib, 1) if ratio_biens_pib is not None else None,
        # Metriques calculees en section 15 (METRIQUES_ANALYSE.csv) mais absentes jusqu'ici
        # des tableaux exportes -- ajoutees ici pour que valeurs_t3 (Tableau 3) les reprenne.
        'ratio_ide_solde_courant': arrondi('RATIO_IDE_SOLDE_COURANT'),
        'ratio_eo_pib': arrondi('RATIO_ERREURS_OMISSIONS'),
        'taux_ouverture': arrondi('TAUX_OUVERTURE_COMMERCIALE'),
        'ide_net': r['IDE_NET'] if pd.notna(r['IDE_NET']) else None,
        'portefeuille_lt_net': r['PORTEFEUILLE_LT_NET'] if pd.notna(r['PORTEFEUILLE_LT_NET']) else None,
        'balance_de_base': r['BALANCE_DE_BASE'] if pd.notna(r['BALANCE_DE_BASE']) else None,
        'pib_usd_millions': pib if pd.notna(pib) else None,
    }


INDICATEURS_PAR_PAYS = {
    code: {annee: indicateurs_cemac(code, annee) for annee in ANNEES_PAR_PAYS[code]}
    for code in PAYS_ELIGIBLES
}


def indicateurs_pour(code, annee):
    return INDICATEURS_PAR_PAYS.get(code, {}).get(annee)


print("Indicateurs (TC/TM/TAS/TFR/TRE + IDE net, portefeuille long terme, balance de base, "
      "ratios PIB et ouverture commerciale) calcules pour tous les couples (pays, annee) disponibles.")


Indicateurs (TC/TM/TAS/TFR/TRE + IDE net, portefeuille long terme, balance de base, ratios PIB et ouverture commerciale) calcules pour tous les couples (pays, annee) disponibles.


In [24]:
import plotly.graph_objects as go


def fmt_cemac(v, mode='musd'):
    """Formate un nombre : 'musd' -> entier signé + espace fine, 'pct' -> 1 décimale, 'ratio' -> 2 décimales virgule."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return 'n.d.'
    NB = ' '
    MIN = '−'
    if mode == 'pct':
        return f'+{v:.1f}{NB}%' if v >= 0 else f'{MIN}{abs(v):.1f}{NB}%'
    if mode == 'ratio':
        return f'{v:.2f}'.replace('.', ',')
    v = int(round(v))
    sign = '+' if v >= 0 else ''
    num = f'{abs(v):,}'.replace(',', NB)
    return f"{sign if v >= 0 else MIN}{num}"


def render_table_cemac(df, title, caption, highlight_rows=None, filename=None, fontsize=13):
    """
    Rendu Plotly d'un tableau (remplace l'ancien rendu matplotlib) : export PNG via
    kaleido, style repris (en-tête bleu, lignes alternées, lignes clés surlignées).
    """
    HEADER_BG = '#00448C'
    HL_COLOR = '#D6E4F0'
    ALT_COLOR = '#F4F7FB'

    row_labels = list(df.index)
    col_labels = list(df.columns)
    n_rows = len(row_labels)
    highlight_rows = set(highlight_rows or [])

    def cellule(i, texte):
        return f'<b>{texte}</b>' if i in highlight_rows else texte

    header_vals = ['<b></b>'] + [f'<b>{c}</b>' for c in col_labels]
    cell_vals = [[cellule(i, lbl) for i, lbl in enumerate(row_labels)]]
    for c in col_labels:
        cell_vals.append([cellule(i, v) for i, v in enumerate(df[c].tolist())])

    fill_colors = []
    for _ in range(len(col_labels) + 1):
        col_fill = [HL_COLOR if i in highlight_rows else (ALT_COLOR if i % 2 == 0 else 'white') for i in range(n_rows)]
        fill_colors.append(col_fill)

    fig = go.Figure(data=[go.Table(
        columnwidth=[2.6] + [1.6] * len(col_labels),
        header=dict(
            values=header_vals, fill_color=HEADER_BG,
            font=dict(color='white', size=fontsize, family='Arial'),
            align=['left'] + ['center'] * len(col_labels), height=34,
        ),
        cells=dict(
            values=cell_vals, fill_color=fill_colors,
            font=dict(color='#1a1a1a', size=fontsize - 1, family='Arial'),
            align=['left'] + ['center'] * len(col_labels), height=28,
            line_color='#C8D0DB',
        ),
    )])

    fig.update_layout(
        title=dict(text=title.replace('\n', '<br>'), x=0.01, xanchor='left',
                    font=dict(size=16, family='Arial', color='#1a1a1a')),
        margin=dict(l=10, r=10, t=80, b=50),
        width=max(760, 210 * len(col_labels) + 320),
        height=max(280, n_rows * 30 + 150),
        paper_bgcolor='white',
        annotations=[dict(
            text=caption.replace('\n', '<br>'), xref='paper', yref='paper', x=0, y=-0.05,
            showarrow=False, font=dict(size=10, color='#555555'), align='left',
        )],
    )

    if filename:
        fig.write_image(str(filename), scale=2)
    return fig


FIGURES_DIR = BASE / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [25]:
rows_t1_cemac = [
    ('Compte courant', 'solde_courant'),
    ('  Biens', 'biens'),
    ('  Services', 'services'),
    ('  Revenu primaire', 'rev_primaire'),
    ('  Revenu secondaire', 'rev_secondaire'),
    ('Compte de capital', 'compte_capital'),
    ('Capacite de financement', 'capacite_finan'),
    ('Compte financier (net)', 'compte_financier'),
    ('Erreurs et omissions', 'erreurs_omissions'),
]

# Tableau 1 complet (tous pays eligibles x toutes leurs annees) -> CSV, valeurs brutes ---
lignes_t1 = []
for code, nom in PAYS_ELIGIBLES.items():
    for a in ANNEES_PAR_PAYS[code]:
        b = bdp_pour(code, a)
        ligne = {'COUNTRY_CODE': code, 'PAYS': nom, 'ANNEE': a}
        ligne.update({cle: b[cle] for _, cle in rows_t1_cemac})
        lignes_t1.append(ligne)

df_t1_complet = pd.DataFrame(lignes_t1)
chemin_t1_csv = BASE / 'data' / 'TABLEAU1_SOLDES_COMPLET.csv'
df_t1_complet.to_csv(chemin_t1_csv, index=False)
print(f"{chemin_t1_csv} : {len(df_t1_complet)} lignes ({df_t1_complet['COUNTRY_CODE'].nunique()} pays x annees) "
      f"-- valeurs en millions USD, source : IMF Statistics API (BOP), calculs de l'auteur.")

# Exemple illustratif : tableau par pays (le premier pays eligible, toutes ses annees) ---
code_ex1, nom_ex1 = next(iter(PAYS_ELIGIBLES.items()))
annees_ex1 = ANNEES_PAR_PAYS[code_ex1]
data_ex1 = {a: [fmt_cemac(bdp_pour(code_ex1, a)[cle]) for _, cle in rows_t1_cemac] for a in annees_ex1}
df_t1_ex1 = pd.DataFrame(data_ex1, index=[r[0] for r in rows_t1_cemac])

fig_t1_ex1 = render_table_cemac(
    df_t1_ex1,
    title=f'Tableau 1 (exemple)  {nom_ex1}\nPrincipaux soldes de la balance des paiements '
          f'({annees_ex1[0]}–{annees_ex1[-1]}, millions USD)',
    caption=f"Exemple illustratif parmi {len(PAYS_ELIGIBLES)} pays -- serie complete dans "
            f"{chemin_t1_csv.name}. Biens/Services/Revenus primaire/secondaire : fiabilite "
            "limitee (cf. section 14bis).",
    highlight_rows=[0, 5, 6, 7],
    filename=FIGURES_DIR / f'tab1_soldes_{code_ex1}.png',
)
fig_t1_ex1.show()

# Exemple illustratif : tableau comparatif (annee la plus recente avec >= 2 pays) ---
annees_comparables_t1 = [a for a in ANNEES_UNION if len([c for c in PAYS_ELIGIBLES if a in ANNEES_PAR_PAYS[c]]) >= 2]
if annees_comparables_t1:
    annee_ex1 = annees_comparables_t1[-1]
    pays_dispo_ex1 = [c for c in PAYS_ELIGIBLES if annee_ex1 in ANNEES_PAR_PAYS[c]]
    data_comp1 = {PAYS_ELIGIBLES[c]: [fmt_cemac(bdp_pour(c, annee_ex1)[cle]) for _, cle in rows_t1_cemac] for c in pays_dispo_ex1}
    df_t1_comp_ex1 = pd.DataFrame(data_comp1, index=[r[0] for r in rows_t1_cemac])

    fig_t1_comp_ex1 = render_table_cemac(
        df_t1_comp_ex1,
        title=f'Tableau 1 comparatif (exemple)  {annee_ex1}\nPrincipaux soldes de la balance des paiements (millions USD)',
        caption=f"Exemple illustratif -- {len(pays_dispo_ex1)} pays disponibles cette annee-la sur "
                f"{len(PAYS_ELIGIBLES)} au total ; serie complete par annee dans {chemin_t1_csv.name}. "
                "Source : IMF Statistics API (BOP), calculs de l'auteur.",
        highlight_rows=[0, 5, 6, 7],
        filename=FIGURES_DIR / f'tab1_soldes_comparatif_{annee_ex1}.png',
    )
    fig_t1_comp_ex1.show()
else:
    print("Aucune annee avec au moins 2 pays disponibles -- pas d'exemple comparatif a afficher.")


m:\Travail\BEAC\balance_de_paiement\data\TABLEAU1_SOLDES_COMPLET.csv : 420 lignes (42 pays x annees) -- valeurs en millions USD, source : IMF Statistics API (BOP), calculs de l'auteur.


In [26]:
rows_t2_cemac = [
    ('Exportations de biens (M USD)', 'exp_biens', 'musd'),
    ('Importations de biens (M USD)', 'imp_biens', 'musd'),
    ('Taux de couverture (TC)', 'tc', 'ratio'),
    ('Taux de couverture des importations (TM)', 'tm', 'ratio'),
    ("Taux d'absorption services (TAS)", 'tas', 'pct_simple'),
    ('Taux de fuite revenu primaire (TFR)', 'tfr', 'pct_simple'),
    ('Taux de retention (TRE)', 'tre', 'pct_simple'),
]


def formatter_t2(vals, cle, mode):
    v = vals.get(cle)
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return 'n.d.'
    if mode == 'musd':
        return fmt_cemac(v, 'musd')
    if mode == 'ratio':
        return f"{v:.2f}".replace('.', ',')
    return f"{v:.1f} %"


# Tableau 2 complet (tous pays eligibles x toutes leurs annees) -> CSV, valeurs brutes ---
lignes_t2 = []
for code, nom in PAYS_ELIGIBLES.items():
    for a in ANNEES_PAR_PAYS[code]:
        vals = {**bdp_pour(code, a), **indicateurs_pour(code, a)}
        ligne = {'COUNTRY_CODE': code, 'PAYS': nom, 'ANNEE': a}
        ligne.update({cle: vals.get(cle) for _, cle, _ in rows_t2_cemac})
        lignes_t2.append(ligne)

df_t2_complet = pd.DataFrame(lignes_t2)
chemin_t2_csv = BASE / 'data' / 'TABLEAU2_TERMES_ECHANGE_COMPLET.csv'
df_t2_complet.to_csv(chemin_t2_csv, index=False)
print(f"{chemin_t2_csv} : {len(df_t2_complet)} lignes ({df_t2_complet['COUNTRY_CODE'].nunique()} pays x annees) "
      "-- source : IMF Statistics API (BOP/WEO), calculs de l'auteur.")

# Exemple illustratif : tableau par pays ---
code_ex2, nom_ex2 = next(iter(PAYS_ELIGIBLES.items()))
annees_ex2 = ANNEES_PAR_PAYS[code_ex2]
data_ex2 = {}
for a in annees_ex2:
    vals = {**bdp_pour(code_ex2, a), **indicateurs_pour(code_ex2, a)}
    data_ex2[a] = [formatter_t2(vals, cle, mode) for _, cle, mode in rows_t2_cemac]
df_t2_ex2 = pd.DataFrame(data_ex2, index=[r[0] for r in rows_t2_cemac])

fig_t2_ex2 = render_table_cemac(
    df_t2_ex2,
    title=f"Tableau 2 (exemple)  {nom_ex2}\nIndicateurs de termes de l'echange et de retention "
          f"des exportations ({annees_ex2[0]}–{annees_ex2[-1]})",
    caption=f"Exemple illustratif parmi {len(PAYS_ELIGIBLES)} pays -- serie complete dans "
            f"{chemin_t2_csv.name}.",
    highlight_rows=[2, 3, 6],
    filename=FIGURES_DIR / f'tab2_termes_echange_{code_ex2}.png',
)
fig_t2_ex2.show()

# Exemple illustratif : tableau comparatif ---
annees_comparables_t2 = [a for a in ANNEES_UNION if len([c for c in PAYS_ELIGIBLES if a in ANNEES_PAR_PAYS[c]]) >= 2]
if annees_comparables_t2:
    annee_ex2 = annees_comparables_t2[-1]
    pays_dispo_ex2 = [c for c in PAYS_ELIGIBLES if annee_ex2 in ANNEES_PAR_PAYS[c]]
    data_comp2 = {}
    for c in pays_dispo_ex2:
        vals = {**bdp_pour(c, annee_ex2), **indicateurs_pour(c, annee_ex2)}
        data_comp2[PAYS_ELIGIBLES[c]] = [formatter_t2(vals, cle, mode) for _, cle, mode in rows_t2_cemac]
    df_t2_comp_ex2 = pd.DataFrame(data_comp2, index=[r[0] for r in rows_t2_cemac])

    fig_t2_comp_ex2 = render_table_cemac(
        df_t2_comp_ex2,
        title=f"Tableau 2 comparatif (exemple)  {annee_ex2}\nIndicateurs de termes de l'echange et de retention des exportations",
        caption=f"Exemple illustratif -- {len(pays_dispo_ex2)} pays disponibles cette annee-la sur "
                f"{len(PAYS_ELIGIBLES)} au total ; serie complete par annee dans {chemin_t2_csv.name}. "
                "Source : IMF Statistics API (BOP/WEO), calculs de l'auteur.",
        highlight_rows=[2, 3, 6],
        filename=FIGURES_DIR / f'tab2_termes_echange_comparatif_{annee_ex2}.png',
    )
    fig_t2_comp_ex2.show()
else:
    print("Aucune annee avec au moins 2 pays disponibles -- pas d'exemple comparatif a afficher.")


m:\Travail\BEAC\balance_de_paiement\data\TABLEAU2_TERMES_ECHANGE_COMPLET.csv : 420 lignes (42 pays x annees) -- source : IMF Statistics API (BOP/WEO), calculs de l'auteur.


In [27]:
def qualite_cemac(code_pays, annee):
    b = bdp_pour(code_pays, annee)
    eo = abs(b['erreurs_omissions'])
    sc = abs(b['solde_courant'])
    ratio = eo / sc * 100 if sc else 0
    if ratio < 10:
        return 'Bonne'
    if ratio < 60:
        return 'Moyenne'
    return 'Faible'


rows_t3_cemac = [
    'Solde courant (M USD)',
    'Solde courant / PIB',
    'Solde biens / PIB',
    'Taux de couverture biens (TC)',
    'Taux de couverture importations (TM)',
    "Taux d'absorption services",
    'Taux de fuite revenu primaire',
    'Taux de retention',
    'Capacite de financement (M USD)',
    'Erreurs & Omissions (M USD)',
    'Qualite statistique',
]

CLES_T3 = ['solde_courant', 'ratio_pib', 'ratio_biens_pib', 'tc', 'tm', 'tas', 'tfr', 'tre',
           'capacite_finan', 'erreurs_omissions', 'qualite']


def valeurs_t3(code, annee):
    b = bdp_pour(code, annee)
    ind = indicateurs_pour(code, annee)
    return {
        'solde_courant': b['solde_courant'],
        'ratio_pib': ind['ratio_pib'],
        'ratio_biens_pib': ind['ratio_biens_pib'],
        'tc': ind['tc'],
        'tm': ind['tm'],
        'tas': ind['tas'],
        'tfr': ind['tfr'],
        'tre': ind['tre'],
        'capacite_finan': b['capacite_finan'],
        'compte_financier': b['compte_financier'],
        # Metriques de la section 15 desormais reprises ici (cf. indicateurs_cemac) --
        # jusque-la calculees mais absentes de tous les tableaux exportes.
        'ide_net': ind['ide_net'],
        'portefeuille_lt_net': ind['portefeuille_lt_net'],
        'balance_de_base': ind['balance_de_base'],
        'ratio_ide_solde_courant': ind['ratio_ide_solde_courant'],
        'erreurs_omissions': b['erreurs_omissions'],
        'ratio_eo_pib': ind['ratio_eo_pib'],
        'taux_ouverture': ind['taux_ouverture'],
        'pib_usd_millions': ind['pib_usd_millions'],
        'qualite': qualite_cemac(code, annee),
    }


def colonne_t3(code, annee):
    v = valeurs_t3(code, annee)
    return [
        fmt_cemac(v['solde_courant']),
        fmt_cemac(v['ratio_pib'], 'pct') if v['ratio_pib'] is not None else 'n.d.',
        fmt_cemac(v['ratio_biens_pib'], 'pct') if v['ratio_biens_pib'] is not None else 'n.d.',
        f"{v['tc']:.2f}".replace('.', ',') if v['tc'] is not None else 'n.d.',
        f"{v['tm']:.2f}".replace('.', ',') if v['tm'] is not None else 'n.d.',
        f"{v['tas']:.1f} %" if v['tas'] is not None else 'n.d.',
        f"{v['tfr']:.1f} %" if v['tfr'] is not None else 'n.d.',
        f"{v['tre']:.1f} %" if v['tre'] is not None else 'n.d.',
        fmt_cemac(v['capacite_finan']),
        fmt_cemac(v['erreurs_omissions']),
        v['qualite'],
    ]


# Tableau 3 complet (tous pays eligibles x toutes leurs annees) -> CSV, valeurs brutes ---
lignes_t3 = []
for code, nom in PAYS_ELIGIBLES.items():
    for a in ANNEES_PAR_PAYS[code]:
        ligne = {'COUNTRY_CODE': code, 'PAYS': nom, 'ANNEE': a}
        ligne.update(valeurs_t3(code, a))
        lignes_t3.append(ligne)

df_t3_complet = pd.DataFrame(lignes_t3)
chemin_t3_csv = BASE / 'data' / 'TABLEAU3_DIAGNOSTIC_COMPLET.csv'
df_t3_complet.to_csv(chemin_t3_csv, index=False)
print(f"{chemin_t3_csv} : {len(df_t3_complet)} lignes ({df_t3_complet['COUNTRY_CODE'].nunique()} pays x annees) "
      "-- source : IMF Statistics API (BOP/WEO), calculs de l'auteur.")

# Exemple illustratif : tableau par pays ---
code_ex3, nom_ex3 = next(iter(PAYS_ELIGIBLES.items()))
annees_ex3 = ANNEES_PAR_PAYS[code_ex3]
data_ex3 = {a: colonne_t3(code_ex3, a) for a in annees_ex3}
df_t3_ex3 = pd.DataFrame(data_ex3, index=rows_t3_cemac)

fig_t3_ex3 = render_table_cemac(
    df_t3_ex3,
    title=f'Tableau 3 (exemple)  {nom_ex3}\nTableau de bord comparatif (diagnostic macroeconomique), '
          f'{annees_ex3[0]}–{annees_ex3[-1]}',
    caption=f"Exemple illustratif parmi {len(PAYS_ELIGIBLES)} pays -- serie complete dans "
            f"{chemin_t3_csv.name}.",
    highlight_rows=[0, 7],
    filename=FIGURES_DIR / f'tab3_diagnostic_{code_ex3}.png',
)
fig_t3_ex3.show()

# Exemple illustratif : tableau comparatif ---
annees_comparables_t3 = [a for a in ANNEES_UNION if len([c for c in PAYS_ELIGIBLES if a in ANNEES_PAR_PAYS[c]]) >= 2]
if annees_comparables_t3:
    annee_ex3 = annees_comparables_t3[-1]
    pays_dispo_ex3 = [c for c in PAYS_ELIGIBLES if annee_ex3 in ANNEES_PAR_PAYS[c]]
    data_comp3 = {PAYS_ELIGIBLES[c]: colonne_t3(c, annee_ex3) for c in pays_dispo_ex3}
    df_t3_comp_ex3 = pd.DataFrame(data_comp3, index=rows_t3_cemac)

    fig_t3_comp_ex3 = render_table_cemac(
        df_t3_comp_ex3,
        title=f'Tableau 3 comparatif (exemple)  {annee_ex3}\nTableau de bord comparatif (diagnostic macroeconomique)',
        caption=f"Exemple illustratif -- {len(pays_dispo_ex3)} pays disponibles cette annee-la sur "
                f"{len(PAYS_ELIGIBLES)} au total ; serie complete par annee dans {chemin_t3_csv.name}. "
                "Source : IMF Statistics API (BOP/WEO), calculs de l'auteur.",
        highlight_rows=[0, 7],
        filename=FIGURES_DIR / f'tab3_diagnostic_comparatif_{annee_ex3}.png',
    )
    fig_t3_comp_ex3.show()
else:
    print("Aucune annee avec au moins 2 pays disponibles -- pas d'exemple comparatif a afficher.")


m:\Travail\BEAC\balance_de_paiement\data\TABLEAU3_DIAGNOSTIC_COMPLET.csv : 420 lignes (42 pays x annees) -- source : IMF Statistics API (BOP/WEO), calculs de l'auteur.


---
#### Tableau 4 -- detail credit/debit par poste (tous pays eligibles x toutes leurs annees)

Complement aux tableaux 1-3 (qui ne montrent que des soldes nets ou des ratios) : credit
et debit bruts pour chaque poste du compte courant, du compte de capital et du compte
financier -- Biens, Services, Revenu primaire, Revenu secondaire, Transferts en capital,
Investissements directs, Investissements de portefeuille, Autres investissements. Meme
perimetre pays/annees que les tableaux precedents (`PAYS_ELIGIBLES`, `ANNEES_PAR_PAYS`).
Source : agregation par SOUS_TYPE_1 de `BOP_PAR_COMPTE.csv` (section 14), colonnes
VALEUR_CREDIT/VALEUR_DEBIT deja calculees en section 15 (variable `niveau1`).


In [28]:
POSTES_DETAIL_CD = [
    ('Compte courant', 'Biens', 'BIENS'),
    ('Compte courant', 'Services', 'SERVICES'),
    ('Compte courant', 'Revenu primaire', 'REVENU_PRIMAIRE'),
    ('Compte courant', 'Revenu secondaire', 'REVENU_SECONDAIRE'),
    ('Compte de capital', 'Transferts en capital', 'TRANSFERTS_CAPITAL'),
    ('Compte financier', 'Investissements directs', 'IDE'),
    ('Compte financier', 'Investissements de portefeuille', 'PORTEFEUILLE'),
    ('Compte financier', 'Autres investissements', 'AUTRES_INVESTISSEMENTS'),
]

niveau1_etude = niveau1[niveau1['COUNTRY_CODE'].isin(PAYS_ELIGIBLES)]


def valeur_credit_debit(code_pays, annee, type_compte, sous_type1, colonne):
    s = niveau1_etude[
        (niveau1_etude['COUNTRY_CODE'] == code_pays) & (niveau1_etude['ANNEE'] == annee)
        & (niveau1_etude['TYPE_DE_COMPTE'] == type_compte) & (niveau1_etude['SOUS_TYPE_1'] == sous_type1)
    ][colonne]
    return s.iloc[0] if len(s) else None


# Tableau 4 complet (tous pays eligibles x toutes leurs annees) -> CSV, credit/debit bruts ---
lignes_t4 = []
for code, nom in PAYS_ELIGIBLES.items():
    for a in ANNEES_PAR_PAYS[code]:
        ligne = {'COUNTRY_CODE': code, 'PAYS': nom, 'ANNEE': a}
        for type_compte, sous_type1, prefixe in POSTES_DETAIL_CD:
            ligne[f'CREDIT_{prefixe}'] = valeur_credit_debit(code, a, type_compte, sous_type1, 'VALEUR_CREDIT')
            ligne[f'DEBIT_{prefixe}'] = valeur_credit_debit(code, a, type_compte, sous_type1, 'VALEUR_DEBIT')
        lignes_t4.append(ligne)

df_t4_complet = pd.DataFrame(lignes_t4)
chemin_t4_csv = BASE / 'data' / 'TABLEAU4_DETAIL_CREDIT_DEBIT_COMPLET.csv'
df_t4_complet.to_csv(chemin_t4_csv, index=False)
print(f"{chemin_t4_csv} : {len(df_t4_complet)} lignes ({df_t4_complet['COUNTRY_CODE'].nunique()} pays x annees) "
      "-- credit/debit bruts par poste (millions USD), source : IMF Statistics API (BOP), calculs de l'auteur.")

# Colonnes formatees Credit/Debit par poste (memes libelles pour l'exemple par pays et l'exemple comparatif) ---
COLONNES_T4 = []
for _, sous_type1, prefixe in POSTES_DETAIL_CD:
    COLONNES_T4.append((f'Credit {sous_type1}', f'CREDIT_{prefixe}'))
    COLONNES_T4.append((f'Debit {sous_type1}', f'DEBIT_{prefixe}'))


def tableau_t4_formate(df_sub, index_series):
    """Sous-ensemble de df_t4_complet -> tableau presentable (colonnes Credit/Debit par poste,
    formatees), avec l'index fourni (annees ou noms de pays) -- pas de transposition : meme
    orientation que le CSV (une ligne par pays x annee)."""
    df_fmt = pd.DataFrame({lbl: df_sub[cle].apply(fmt_cemac).values for lbl, cle in COLONNES_T4})
    df_fmt.index = index_series.values
    return df_fmt


# Exemple illustratif : tableau par pays (annees en lignes, postes en colonnes) ---
code_ex4, nom_ex4 = next(iter(PAYS_ELIGIBLES.items()))
annees_ex4 = ANNEES_PAR_PAYS[code_ex4]
sub_ex4 = df_t4_complet[df_t4_complet['COUNTRY_CODE'] == code_ex4].sort_values('ANNEE')
df_t4_ex4 = tableau_t4_formate(sub_ex4, sub_ex4['ANNEE'])

fig_t4_ex4 = render_table_cemac(
    df_t4_ex4,
    title=f'Tableau 4 (exemple)  {nom_ex4}\nDetail credit/debit par poste ({annees_ex4[0]}–{annees_ex4[-1]}, millions USD)',
    caption=f"Exemple illustratif parmi {len(PAYS_ELIGIBLES)} pays -- serie complete dans "
            f"{chemin_t4_csv.name}.",
    fontsize=10,
    filename=FIGURES_DIR / f'tab4_detail_credit_debit_{code_ex4}.png',
)
fig_t4_ex4.show()

# Exemple illustratif : tableau comparatif (pays en lignes, postes en colonnes) ---
annees_comparables_t4 = [a for a in ANNEES_UNION if len([c for c in PAYS_ELIGIBLES if a in ANNEES_PAR_PAYS[c]]) >= 2]
if annees_comparables_t4:
    annee_ex4 = annees_comparables_t4[-1]
    sub_comp4 = df_t4_complet[df_t4_complet['ANNEE'] == annee_ex4].sort_values('PAYS')
    df_t4_comp4 = tableau_t4_formate(sub_comp4, sub_comp4['PAYS'])

    fig_t4_comp4 = render_table_cemac(
        df_t4_comp4,
        title=f'Tableau 4 comparatif (exemple)  {annee_ex4}\nDetail credit/debit par poste (millions USD)',
        caption=f"Exemple illustratif -- {len(sub_comp4)} pays disponibles cette annee-la sur "
                f"{len(PAYS_ELIGIBLES)} au total ; serie complete par annee dans {chemin_t4_csv.name}.",
        fontsize=10,
        filename=FIGURES_DIR / f'tab4_detail_credit_debit_comparatif_{annee_ex4}.png',
    )
    fig_t4_comp4.show()
else:
    print("Aucune annee avec au moins 2 pays disponibles -- pas d'exemple comparatif a afficher.")


m:\Travail\BEAC\balance_de_paiement\data\TABLEAU4_DETAIL_CREDIT_DEBIT_COMPLET.csv : 420 lignes (42 pays x annees) -- credit/debit bruts par poste (millions USD), source : IMF Statistics API (BOP), calculs de l'auteur.
